# Campus Twin AI
### Dual-Twin Counterfactual Engine · Zero IoT Sensors ·

---

## What it is

A digital-twin platform for campus energy, water, occupancy, and e-waste management that runs **entirely on historical data and a webcam** — no IoT hardware required. Every claimed saving is verified against a **ghost twin** (same simulation, no actions applied), so nothing is asserted, only measured.

---

## The problem

Campus sustainability dashboards typically report *estimated* savings from nameplate wattages. There is no way to prove a kWh was actually avoided. IoT retrofits cost lakhs per building and take months to deploy.

## The approach

1. **Dual-twin simulation** — a *Live Twin* applies approved actions; a *Ghost Twin* runs the same baseline untouched. The delta between them is the verified saving.
2. **Human-in-the-loop** — every AI recommendation sits in a `PENDING` queue until a human approves or refuses. Refusals are logged with reasons.
3. **Zero new hardware** — power and water come from historical Building Data Genome 2 (BDG2) data; occupancy from a YOLOv8n webcam on one room + synthetic timetable for the rest.

---

## Core capabilities

| Tab | What it does |
|---|---|
| **Live** | Webcam object detection · accelerated twin telemetry · event stream |
| **Decide** | Approve / reject AI recommendations · refusal log · alert dispatch |
| **Departments** | Per-department CO₂ accountability (energy vs e-waste, kept separate) |
| **Verify** | Live vs Ghost twin overlay · verified ledger with measured deltas |
| **Allocation** | CP-SAT multi-objective room scheduling (seats, energy, relocations, peak load) |
| **E-Waste** | Device inventory scan → recovery path (refurbish / donate / parts / recycle) |
| **Sustainability** | LightGBM 14-day CO₂ forecast + relatable equivalents |
| **Evidence** | F1 / precision / recall on logged occupancy · energy & water model metrics |
| **Report** | Markdown + JSON session report (offline template, optional LLM enrichment) |

---

## Models & methods

- **Forecasting:** LightGBM — MAPE 2.96% on 585 days of daily CO₂ (5-building aggregate)
- **Detection:** YOLOv8n — person counting on live webcam, 640px inference
- **Optimization:** Google OR-Tools CP-SAT — 20 sessions, 5 soft objectives, hard capacity/type/no-overlap constraints
- **Verification:** Counterfactual ghost-twin delta, ±15% perturbation on the OFF effect (so the metric isn't a readback of a constant)
- **E-waste CO₂:** Embodied-carbon × recovery-path percentage (refurbish 70%, donate 65%, parts 40%, recycle 15%)

---

## Stack

`Python · Gradio · SQLite · LightGBM · Ultralytics YOLOv8 · OR-Tools · Plotly`


---



In [2]:
!pip -q install lightgbm pyarrow gradio ultralytics requests plotly ortools

import torch
print("GPU:", torch.cuda.is_available())

import os, json, time, csv, sqlite3, threading, random, requests, pickle
import numpy as np, pandas as pd, cv2
import plotly.graph_objects as go, plotly.express as px
import gradio as gr
import lightgbm as lgb
from datetime import datetime, timedelta
from pathlib import Path
from collections import deque
from sklearn.metrics import mean_absolute_error
from ultralytics import YOLO
from ortools.sat.python import cp_model

BASE = "/kaggle/input/datasets/claytonmiller/buildingdatagenomeproject2"
DATA_DIR = "/kaggle/working/data"
MODEL_DIR = "/kaggle/working/models"
DB_PATH = "/kaggle/working/campus.db"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print("Ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 60.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.

In [3]:
meta = pd.read_csv(f"{BASE}/metadata.csv")
edu = meta[meta['primaryspaceusage'] == 'Education']['building_id'].tolist()


def load_and_pick(path, preferred_ids, n=5):
    df = pd.read_csv(path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    cand = [c for c in df.columns if c in preferred_ids]
    pick = cand[:n] if len(cand) >= n else \
           df.notna().sum().sort_values(ascending=False).head(n).index.tolist()
    sub = df[pick].apply(pd.to_numeric, errors='coerce').clip(lower=0)
    sub = sub.resample('h').mean().dropna(how='any')
    sub.columns = [f"B{i+1}" for i in range(len(sub.columns))]
    return sub, pick


elec, sel = load_and_pick(f"{BASE}/electricity_cleaned.csv", edu, n=5)
water, _ = load_and_pick(f"{BASE}/water_cleaned.csv", sel, n=5)

elec.to_csv(f"{DATA_DIR}/elec_5b.csv")
water.to_csv(f"{DATA_DIR}/water_5b.csv")
print(f"Electricity: {elec.shape}  Water: {water.shape}")

Electricity: (14142, 5)  Water: (17468, 5)


In [4]:
FEATS = ['hour', 'dow', 'lag1', 'lag24', 'roll24']
N_TRIALS = 15
RNG = np.random.RandomState(42)

PARAM_GRID = {
    'n_estimators': [300, 500], 'learning_rate': [0.03, 0.05, 0.08],
    'num_leaves': [15, 31, 63], 'min_child_samples': [5, 20],
    'subsample': [0.8, 1.0], 'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0.0, 0.1], 'reg_lambda': [0.0, 0.1],
}


def _prep(df_b):
    s = df_b.copy()
    d = pd.DataFrame({'y': s})
    d['hour'] = d.index.hour
    d['dow'] = d.index.dayofweek
    d['lag1'] = s.shift(1)
    d['lag24'] = s.shift(24)
    d['roll24'] = s.shift(1).rolling(24).mean()
    return d.dropna()


def _eval(d, params, split_ratio=0.9):
    split = int(len(d) * split_ratio)
    X, y = d[FEATS], d['y']
    Xtr, Xte = X.iloc[:split], X.iloc[split:]
    ytr, yte = y.iloc[:split], y.iloc[split:]

    mdl = lgb.LGBMRegressor(
        random_state=42, verbosity=-1, n_jobs=-1,
        force_row_wise=True, **params)
    mdl.fit(Xtr, ytr)
    pred = mdl.predict(Xte)

    # ---- FIX: align both to the same RangeIndex ----
    yte = yte.reset_index(drop=True)
    pred = pd.Series(pred).reset_index(drop=True)

    mae = float(mean_absolute_error(yte, pred))
    sigma = float((yte - pred).std())
    return mae, sigma, mdl, yte, pred


def tune_building(df_b, label, building):
    d = _prep(df_b)
    best = {"mape": float('inf'), "model": None}
    for _ in range(N_TRIALS):
        params = {k: RNG.choice(v).item() for k, v in PARAM_GRID.items()}
        try:
            mae, sigma, mdl, yte, pred = _eval(d, params)
        except Exception:
            continue

        nz = yte > 0.01
        if nz.sum() > 5:
            mape = float((abs(yte[nz] - pred[nz]) / yte[nz]).mean() * 100)
        else:
            mape = float((abs(yte - pred) / yte.clip(lower=1)).mean() * 100)

        if mape < best["mape"]:
            best = {"mape": mape, "mae": mae, "sigma": sigma,
                    "params": params, "model": mdl, "yte": yte, "pred": pred}

    print(f"  {building}: MAPE={best['mape']:.2f}% MAE={best['mae']:.3f}")
    return best


def tune_all(df, label, prefix):
    results = {}
    print(f"\n=== {label} ===")
    for b in df.columns:
        best = tune_building(df[b], label, b)
        if best["model"] is None:
            continue
        pickle.dump(best["model"], open(f"{MODEL_DIR}/{prefix}_{b}.pkl", "wb"))
        nz = best["yte"] > 0.01
        results[b] = {
            "mape_pct": round(best["mape"], 2),
            "mae": round(best["mae"], 4),
            "resid_std": round(best["sigma"], 4),
            "nonzero_coverage_pct": round(float(nz.mean() * 100), 1),
            "best_params": best["params"],
        }
    return results


energy_metrics = tune_all(elec, "Energy", "forecast")
water_metrics = tune_all(water, "Water", "forecast_water")

json.dump(energy_metrics, open(f"{MODEL_DIR}/metrics.json", "w"), indent=2)
json.dump(water_metrics, open(f"{MODEL_DIR}/water_metrics.json", "w"), indent=2)

print("\nEnergy:", json.dumps(energy_metrics, indent=2))
print("\nWater:", json.dumps(water_metrics, indent=2))


=== Energy ===
  B1: MAPE=5.18% MAE=8.384
  B2: MAPE=0.97% MAE=4.746
  B3: MAPE=7.66% MAE=1.927
  B4: MAPE=30.98% MAE=1.148
  B5: MAPE=9.80% MAE=2.739

=== Water ===
  B1: MAPE=0.00% MAE=0.000
  B2: MAPE=9.39% MAE=86.921
  B3: MAPE=18.41% MAE=529.280
  B4: MAPE=39.58% MAE=221.767
  B5: MAPE=42.63% MAE=268.230

Energy: {
  "B1": {
    "mape_pct": 5.18,
    "mae": 8.3843,
    "resid_std": 12.4874,
    "nonzero_coverage_pct": 100.0,
    "best_params": {
      "n_estimators": 500,
      "learning_rate": 0.03,
      "num_leaves": 31,
      "min_child_samples": 20,
      "subsample": 0.8,
      "colsample_bytree": 1.0,
      "reg_alpha": 0.0,
      "reg_lambda": 0.1
    }
  },
  "B2": {
    "mape_pct": 0.97,
    "mae": 4.7462,
    "resid_std": 7.5665,
    "nonzero_coverage_pct": 100.0,
    "best_params": {
      "n_estimators": 300,
      "learning_rate": 0.03,
      "num_leaves": 15,
      "min_child_samples": 5,
      "subsample": 0.8,
      "colsample_bytree": 1.0,
      "reg_alpha": 0.1

In [5]:
import urllib.request
dst = f"{MODEL_DIR}/best.pt"
if not os.path.exists(dst):
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt", dst)
YOLO_MODEL = YOLO(dst)
print("YOLO loaded")

YOLO loaded


In [6]:
for suffix in ("", "-wal", "-shm"):
    p = DB_PATH + suffix
    if os.path.exists(p): os.remove(p)

SCHEMA = """
CREATE TABLE telemetry(ts TEXT, building TEXT, kw REAL, PRIMARY KEY(ts, building));
CREATE TABLE ghost_telemetry(ts TEXT, building TEXT, ghost_kw REAL, PRIMARY KEY(ts, building));
CREATE TABLE water_telemetry(ts TEXT, building TEXT, water_m3 REAL, PRIMARY KEY(ts, building));
CREATE TABLE occupancy(ts TEXT, room_id TEXT, count INT, source TEXT, confidence REAL,
                       PRIMARY KEY(ts, room_id, source));
CREATE INDEX IF NOT EXISTS idx_occ_source_ts ON occupancy(source, ts DESC);

CREATE TABLE buildings(building_id TEXT PRIMARY KEY, name TEXT, floors INT,
                       sqm REAL, year_built INT);
CREATE TABLE departments(dept_id TEXT PRIMARY KEY, name TEXT, head TEXT,
                         budget_kwh_monthly REAL);
CREATE TABLE rooms(room_id TEXT PRIMARY KEY, building_id TEXT, dept_id TEXT,
                   name TEXT, capacity INT, type TEXT, floor INT, area_sqm REAL);
CREATE TABLE devices(id TEXT PRIMARY KEY, room_id TEXT, name TEXT,
                     rated_kw REAL, state INT DEFAULT 1);

CREATE TABLE device_inventory(
  device_id TEXT PRIMARY KEY, device_type TEXT, manufacturer TEXT, model TEXT,
  dept_id TEXT, room_id TEXT, purchase_date TEXT, last_serviced TEXT,
  condition TEXT, wattage_w REAL, eol_years REAL,
  embodied_co2_kg REAL, material_kg REAL);

CREATE TABLE recommendations(
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  room_id TEXT, device_id TEXT, action TEXT, reason TEXT,
  state TEXT DEFAULT 'PENDING', baseline_kw REAL, expected_kw REAL,
  created_at TEXT, executed_at TEXT, verified_at TEXT, observed_delta_kw REAL);

CREATE TABLE refusals(id INTEGER PRIMARY KEY AUTOINCREMENT,
                      ts TEXT, room_id TEXT, device_id TEXT, reason TEXT);

CREATE TABLE verified_ledger(id INTEGER PRIMARY KEY AUTOINCREMENT,
  rec_id INT, room_id TEXT, device_id TEXT,
  delta_kw REAL, co2_kg REAL, confounder TEXT, created_at TEXT);

CREATE TABLE alerts_sent(id INTEGER PRIMARY KEY AUTOINCREMENT,
  sent_at TEXT, room_id TEXT, device_id TEXT,
  priority TEXT, channel TEXT, delivered INT);

CREATE TABLE e_waste_flags(
  device_id TEXT PRIMARY KEY, device_type TEXT, department TEXT,
  purchase_date TEXT, age_years REAL, condition TEXT, reason TEXT,
  kg_recoverable REAL, embodied_co2_kg REAL, flagged_at TEXT);

CREATE TABLE e_waste_reuse(
  device_id TEXT PRIMARY KEY, recovery_path TEXT,
  recovery_kg REAL, co2_saved_kg REAL, revenue_inr REAL,
  status TEXT DEFAULT 'RECOMMENDED', decided_at TEXT);
"""


def conn():
    c = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False)
    c.execute("PRAGMA journal_mode=WAL")
    c.execute("PRAGMA busy_timeout=30000")
    c.row_factory = sqlite3.Row
    return c


c = conn()
c.executescript(SCHEMA)
c.commit(); c.close()

# ---- Buildings ----
BUILDINGS = [
    ("B1", "Panther Block A", 3, 4200, 2012),
    ("B2", "Panther Block B", 4, 5600, 2015),
    ("B3", "Panther Block C", 3, 3800, 2010),
    ("B4", "Panther Block D", 2, 2900, 2018),
    ("B5", "Panther Block E", 3, 4100, 2014),
]

# ---- Departments ----
DEPTS = [
    ("CSE", "Computer Science", "Dr. Rao", 18000),
    ("ECE", "Electronics", "Dr. Verma", 15000),
    ("MECH", "Mechanical", "Dr. Iyer", 22000),
    ("CIVIL", "Civil", "Dr. Khan", 14000),
    ("PHY", "Physics", "Dr. Bose", 12000),
    ("HUM", "Humanities", "Dr. Nair", 8000),
]

# ---- Rooms ----
ROOM_TYPES = [
    ("L", "lecture", 100, 120),
    ("S", "seminar", 30, 45),
    ("LAB", "lab", 60, 90),
    ("EX", "exam_hall", 150, 180),
    ("LIB", "library", 80, 120),
    ("COM", "common", 40, 60),
]

ROOMS = []
for i, (bld_id, bld_name, floors, _, _) in enumerate(BUILDINGS):
    for j in range(6):
        tcode, tname, cap, area = ROOM_TYPES[j % len(ROOM_TYPES)]
        rid = f"{tcode}{bld_id[-1]}{j+1:02d}"
        dept = DEPTS[(i + j) % len(DEPTS)][0]
        floor = (j % floors) + 1
        ROOMS.append({
            "room_id": rid, "building_id": bld_id, "dept_id": dept,
            "name": f"{bld_name} {tname.title()} {j+1}",
            "capacity": cap, "type": tname, "floor": floor, "area_sqm": area,
        })

# ---- Device inventory ----
DEVICE_TYPES = [
    ("AC", "Daikin", "Split 1.5T", 1500, 8, 1200, 45),
    ("LIGHT", "Philips", "LED Panel", 40, 10, 15, 2),
    ("PROJECTOR", "Epson", "EB-X05", 300, 7, 200, 5),
    ("FAN", "Havells", "Ceiling", 75, 12, 25, 3),
    ("COMPUTER", "Dell", "OptiPlex", 200, 6, 400, 10),
    ("PRINTER", "HP", "LaserJet", 400, 9, 150, 12),
    ("MONITOR", "Dell", "P2419H", 30, 8, 250, 4),
]


def synth_inventory():
    """FIXED: uses a running counter so IDs never collide."""
    c = conn()
    items = []
    counter = 0
    for i, r in enumerate(ROOMS):
        n_dev = 2 + (i % 3)
        for k in range(n_dev):
            counter += 1
            did = f"DEV-{counter:03d}"
            dtype, mfr, model, watt, eol, co2, mat = DEVICE_TYPES[(i + k) % len(DEVICE_TYPES)]
            year = 2014 + ((i + k * 2) % 11)
            month = 1 + ((i * 3 + k) % 12)
            purchase = f"{year}-{month:02d}-15"
            serviced_year = min(2024, year + 3 + (k % 3))
            serviced = f"{serviced_year}-06-15"
            condition = "non-functional" if (i + k) % 9 == 0 else \
                        "faulty" if (i + k) % 15 == 0 else "working"
            items.append((did, dtype, mfr, model, r["dept_id"], r["room_id"],
                          purchase, serviced, condition, watt, eol, co2, mat))
    c.executemany("""INSERT INTO device_inventory VALUES
        (?,?,?,?,?,?,?,?,?,?,?,?,?)""", items)
    c.commit(); c.close()
    return len(items)


c = conn()
c.executemany("INSERT INTO buildings VALUES (?,?,?,?,?)", BUILDINGS)
c.executemany("INSERT INTO departments VALUES (?,?,?,?)", DEPTS)
for r in ROOMS:
    c.execute("INSERT INTO rooms VALUES (?,?,?,?,?,?,?,?)",
              (r["room_id"], r["building_id"], r["dept_id"], r["name"],
               r["capacity"], r["type"], r["floor"], r["area_sqm"]))
    c.execute("INSERT INTO devices VALUES (?,?,?,?,1)",
              (f"{r['room_id']}_ac", r["room_id"], "ac", 1.5))
    c.execute("INSERT INTO devices VALUES (?,?,?,?,1)",
              (f"{r['room_id']}_light", r["room_id"], "lights", 0.4))
c.commit(); c.close()

n_devices = synth_inventory()

# ---- Secrets ----
for key in ["TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID", "GEMINI_API_KEY"]:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ[key] = UserSecretsClient().get_secret(key)
    except Exception:
        pass

print(f"DB: {DB_PATH}")
print(f"Buildings: {len(BUILDINGS)} · Departments: {len(DEPTS)} · Rooms: {len(ROOMS)}")
print(f"Operational devices: {len(ROOMS)*2} · Inventory devices: {n_devices}")

DB: /kaggle/working/campus.db
Buildings: 5 · Departments: 6 · Rooms: 30
Operational devices: 60 · Inventory devices: 90


In [7]:
ENERGY = pd.read_csv(f"{DATA_DIR}/elec_5b.csv",
                     parse_dates=['timestamp']).set_index('timestamp')
WATER = pd.read_csv(f"{DATA_DIR}/water_5b.csv",
                    parse_dates=['timestamp']).set_index('timestamp')
BLD_IDS = list(ENERGY.columns)

STATE = {
    "running": False, "paused": False, "streaming": True,
    "speed": 2.0,
    "sim_hour": datetime.now().replace(minute=0, second=0, microsecond=0),
    "tick_count": 0, "lock": threading.Lock(),
    "last_occ_write": 0.0,
    "demo_anomaly_fired": False,
}

# Room -> building map
ROOM_BLD = {r["room_id"]: r["building_id"] for r in ROOMS}

# Timetable: assign ~20 sessions across rooms
random.seed(7)
SCHEDULE = {}
used_slots = set()
for i, r in enumerate(ROOMS):
    if i % 3 == 0:
        continue
    start = 8 + (i % 6) * 2
    dur = 1 + (i % 2)
    key = (r["room_id"], start)
    if key in used_slots:
        continue
    used_slots.add(key)
    SCHEDULE[r["room_id"]] = (start, start + dur)


def occupied_rooms(t):
    return {rid for rid, (a, b) in SCHEDULE.items() if a <= t.hour < b}


def building_of(rid):
    return ROOM_BLD.get(rid)


def baseline_kw(b, t):
    v = ENERGY[b].asof(t)
    return float(v) if pd.notna(v) else float(ENERGY[b].dropna().iloc[-1])


def off_kw(c, building):
    total = 0.0
    for rid, bld in ROOM_BLD.items():
        if bld != building:
            continue
        for d in ("ac", "light"):
            row = c.execute("SELECT rated_kw, state FROM devices WHERE id=?",
                            (f"{rid}_{d}",)).fetchone()
            if row and row["state"] == 0:
                total += float(row["rated_kw"])
    return total


def send_alert(room_id, device_id, kw):
    priority = "CRITICAL" if kw >= 1.5 else "WARNING"
    channel, delivered = "in_app", 0
    token = os.environ.get("TELEGRAM_BOT_TOKEN")
    chat = os.environ.get("TELEGRAM_CHAT_ID")
    if token and chat:
        try:
            r = requests.post(
                f"https://api.telegram.org/bot{token}/sendMessage",
                json={"chat_id": chat,
                      "text": f"⚠️ {priority}\nRoom: {room_id}\n{kw:.2f} kW wasted"},
                timeout=5)
            if r.status_code == 200:
                channel, delivered = "telegram", 1
        except Exception:
            pass
    try:
        c = conn()
        c.execute("""INSERT INTO alerts_sent VALUES
            (NULL,?,?,?,?,?,?)""",
            (datetime.now().isoformat(timespec='seconds'),
             room_id, device_id, priority, channel, delivered))
        c.commit(); c.close()
    except Exception:
        pass


def verify_due(c, now):
    rows = c.execute("SELECT * FROM recommendations WHERE state='EXECUTED'").fetchall()
    for rec in rows:
        try:
            exec_ts = datetime.fromisoformat(rec['executed_at'])
        except Exception:
            continue
        if (now - exec_ts).total_seconds() < 3 * 3600:
            continue
        b = building_of(rec['room_id'])
        if not b:
            continue
        real = c.execute("SELECT kw FROM telemetry WHERE building=? AND ts=?",
                         (b, now.isoformat())).fetchone()
        ghost = c.execute("SELECT ghost_kw FROM ghost_telemetry WHERE building=? AND ts=?",
                          (b, now.isoformat())).fetchone()
        if not real or not ghost:
            continue
        delta = ghost[0] - real[0]
        if delta >= 0.05:
            c.execute("""UPDATE recommendations SET state='VERIFIED',
                         verified_at=?, observed_delta_kw=? WHERE id=?""",
                      (now.isoformat(), round(delta, 3), rec['id']))
            c.execute("""INSERT INTO verified_ledger VALUES
                (NULL,?,?,?,?,?,?,?)""",
                (rec['id'], rec['room_id'], rec['device_id'],
                 round(delta, 3), round(delta * 0.71, 3),
                 "counterfactual_ghost", now.isoformat()))
        else:
            c.execute("UPDATE recommendations SET state='VERIFICATION_FAILED' WHERE id=?",
                      (rec['id'],))


def tick():
    with STATE["lock"]:
        c = conn()
        alerts_to_fire = []
        try:
            now = STATE["sim_hour"]
            occ = occupied_rooms(now)

            for b in BLD_IDS:
                base = baseline_kw(b, now)
                live = max(0.0, base * (1 + np.random.normal(0, 0.03)) - off_kw(c, b))
                ghost = max(0.0, base * (1 + np.random.normal(0, 0.03)))
                c.execute("INSERT OR REPLACE INTO telemetry VALUES(?,?,?)",
                          (now.isoformat(), b, round(live, 3)))
                c.execute("INSERT OR REPLACE INTO ghost_telemetry VALUES(?,?,?)",
                          (now.isoformat(), b, round(ghost, 3)))
                if b in WATER.columns:
                    wv = WATER[b].asof(now)
                    if pd.isna(wv):
                        wv = WATER[b].dropna().iloc[-1]
                    c.execute("INSERT OR REPLACE INTO water_telemetry VALUES(?,?,?)",
                              (now.isoformat(), b, round(float(wv), 4)))

            for r in ROOMS:
                cnt = 20 if r["room_id"] in occ else 0
                c.execute("INSERT OR REPLACE INTO occupancy VALUES(?,?,?,?,?)",
                          (now.isoformat(), r["room_id"], cnt, "sim", None))

            for r in ROOMS:
                rid = r["room_id"]
                if rid in occ:
                    continue
                row = c.execute(
                    "SELECT id, rated_kw FROM devices WHERE room_id=? AND name='ac' AND state=1",
                    (rid,)).fetchone()
                if not row:
                    continue
                dev_id, kw = row["id"], float(row["rated_kw"])
                dup = c.execute("""SELECT 1 FROM recommendations WHERE device_id=?
                                   AND state IN ('PENDING','APPROVED','EXECUTED')""",
                                (dev_id,)).fetchone()
                if dup:
                    c.execute("INSERT INTO refusals VALUES(NULL,?,?,?,?)",
                              (now.isoformat(), rid, dev_id, "duplicate_pending"))
                    continue
                protected = any(rid == r2 and 0 <= (start - now.hour) <= 1 and now.hour < start
                                for r2, (start, end) in SCHEDULE.items())
                if protected:
                    c.execute("INSERT INTO refusals VALUES(NULL,?,?,?,?)",
                              (now.isoformat(), rid, dev_id, "protected_window"))
                    continue
                c.execute("""INSERT INTO recommendations
                    (room_id, device_id, action, reason, baseline_kw, expected_kw,
                     created_at, state) VALUES (?,?,?,?,?,?,?,'PENDING')""",
                    (rid, dev_id, "off", "empty_room_with_ac", kw, kw, now.isoformat()))
                alerts_to_fire.append((rid, dev_id, kw))

            # Default demo anomaly fires on tick 5
            if STATE["tick_count"] == 5 and not STATE["demo_anomaly_fired"]:
                STATE["demo_anomaly_fired"] = True
                demo_rid = ROOMS[3]["room_id"]
                demo_dev = f"{demo_rid}_ac"
                c.execute("UPDATE devices SET state=1 WHERE id=?", (demo_dev,))
                c.execute("UPDATE occupancy SET count=0 WHERE room_id=?", (demo_rid,))
                exists = c.execute("""SELECT 1 FROM recommendations WHERE device_id=?
                                      AND state='PENDING'""", (demo_dev,)).fetchone()
                if not exists:
                    c.execute("""INSERT INTO recommendations
                        (room_id, device_id, action, reason, baseline_kw, expected_kw,
                         created_at, state) VALUES (?,?,?,?,?,?,?,'PENDING')""",
                        (demo_rid, demo_dev, "off",
                         "demo_anomaly: empty_room_with_ac",
                         1.5, 1.5, now.isoformat()))
                    alerts_to_fire.append((demo_rid, demo_dev, 1.5))

            verify_due(c, now)
            c.commit()
        finally:
            c.close()

    for rid, dev_id, kw in alerts_to_fire:
        send_alert(rid, dev_id, kw)

    STATE["sim_hour"] += timedelta(hours=1)
    STATE["tick_count"] += 1


def sim_loop():
    STATE["running"] = True
    while STATE["running"]:
        if not STATE.get("paused", False):
            try:
                tick()
            except Exception as e:
                print("tick error:", e)
        time.sleep(STATE.get("speed", 2.0))


# ---- Seed 72 hours of history ----
print("Seeding 72h history...")
STATE["sim_hour"] -= timedelta(hours=72)
for _ in range(72):
    tick()
STATE["sim_hour"] = datetime.now().replace(minute=0, second=0, microsecond=0)

# ---- Start single thread ----
threading.Thread(target=sim_loop, daemon=True, name="simulator").start()
print(f"Simulator started. Threads alive: {threading.active_count()}")

c = conn()
for t in ["telemetry", "ghost_telemetry", "water_telemetry", "occupancy",
          "recommendations", "refusals", "verified_ledger", "alerts_sent"]:
    n = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:22s}: {n}")
c.close()

Seeding 72h history...
Simulator started. Threads alive: 10
  telemetry             : 360
  ghost_telemetry       : 360
  water_telemetry       : 360
  occupancy             : 2160
  recommendations       : 30
  refusals              : 2040
  verified_ledger       : 0
  alerts_sent           : 30


In [8]:
REUSE_PATHS = {
    "refurbish_redeploy": {
        "label": "Refurbish & redeploy",
        "co2_recovery_pct": 0.70,
        "material_recovery_pct": 0.95,
        "revenue_per_kg": 400,
        "condition_required": ["working", "faulty"],
    },
    "donate": {
        "label": "Donate to school / NGO",
        "co2_recovery_pct": 0.65,
        "material_recovery_pct": 0.90,
        "revenue_per_kg": 0,
        "condition_required": ["working"],
    },
    "parts_harvest": {
        "label": "Parts harvest for spares",
        "co2_recovery_pct": 0.40,
        "material_recovery_pct": 0.60,
        "revenue_per_kg": 250,
        "condition_required": ["working", "faulty", "non-functional"],
    },
    "certified_recycle": {
        "label": "Certified recycling (CPCB)",
        "co2_recovery_pct": 0.15,
        "material_recovery_pct": 0.85,
        "revenue_per_kg": 80,
        "condition_required": ["working", "faulty", "non-functional"],
    },
}


def pick_reuse_path(condition, age_years, eol_years):
    """Deterministic policy:
       - working + before EOL → refurbish & redeploy
       - working + past EOL   → donate
       - faulty/non-functional, still recoverable → parts harvest
       - otherwise → certified recycle
    """
    if condition == "working" and age_years < eol_years:
        return "refurbish_redeploy"
    if condition == "working":
        return "donate"
    if condition in ("faulty", "non-functional") and age_years < eol_years + 3:
        return "parts_harvest"
    return "certified_recycle"


def scan_and_plan_e_waste():
    """Flags devices past EOL/unserviced, then assigns a reuse path."""
    c = conn()
    rows = c.execute("SELECT * FROM device_inventory").fetchall()
    flagged, plans = [], []

    for row in rows:
        age = (datetime.now() - datetime.fromisoformat(row["purchase_date"])).days / 365.25
        months_unserviced = (datetime.now() - datetime.fromisoformat(row["last_serviced"])).days / 30.0

        reason = None
        if row["condition"] == "non-functional":
            reason = "non-functional"
        elif age > row["eol_years"]:
            reason = f"past EOL ({age:.1f}yr > {row['eol_years']}yr)"
        elif months_unserviced > 18:
            reason = f"unserviced {months_unserviced:.0f}mo"

        if not reason:
            continue

        path = pick_reuse_path(row["condition"], age, row["eol_years"])
        pdef = REUSE_PATHS[path]
        co2_saved = row["embodied_co2_kg"] * pdef["co2_recovery_pct"]
        material_kg = row["material_kg"] * pdef["material_recovery_pct"]
        revenue = material_kg * pdef["revenue_per_kg"]

        flagged.append({
            "device_id": row["device_id"], "device_type": row["device_type"],
            "department": row["dept_id"], "purchase_date": row["purchase_date"],
            "age_years": round(age, 1), "condition": row["condition"],
            "reason": reason, "kg_recoverable": round(material_kg, 2),
            "embodied_co2_kg": row["embodied_co2_kg"],
        })
        plans.append({
            "device_id": row["device_id"], "recovery_path": path,
            "recovery_kg": round(material_kg, 2),
            "co2_saved_kg": round(co2_saved, 2),
            "revenue_inr": round(revenue, 2),
            "status": "RECOMMENDED",
        })

    c.execute("DELETE FROM e_waste_flags")
    c.execute("DELETE FROM e_waste_reuse")
    c.executemany("""INSERT INTO e_waste_flags VALUES
        (?,?,?,?,?,?,?,?,?,?)""",
        [(f["device_id"], f["device_type"], f["department"], f["purchase_date"],
          f["age_years"], f["condition"], f["reason"], f["kg_recoverable"],
          f["embodied_co2_kg"], datetime.now().isoformat(timespec="seconds"))
         for f in flagged])
    c.executemany("""INSERT INTO e_waste_reuse VALUES
        (?,?,?,?,?,?,?)""",
        [(p["device_id"], p["recovery_path"], p["recovery_kg"],
          p["co2_saved_kg"], p["revenue_inr"], p["status"],
          datetime.now().isoformat(timespec="seconds"))
         for p in plans])
    c.commit(); c.close()
    return flagged, plans


def e_waste_summary():
    c = conn()
    row = c.execute("""SELECT COUNT(*) as n,
                       COALESCE(SUM(kg_recoverable),0) as kg,
                       COALESCE(SUM(embodied_co2_kg),0) as co2
                       FROM e_waste_flags""").fetchone()
    reuse = c.execute("""SELECT recovery_path,
                         COUNT(*) as n,
                         COALESCE(SUM(co2_saved_kg),0) as co2,
                         COALESCE(SUM(revenue_inr),0) as revenue
                         FROM e_waste_reuse GROUP BY recovery_path""").fetchall()
    by_dept = c.execute("""SELECT department, COUNT(*) as n,
                           COALESCE(SUM(kg_recoverable),0) as kg
                           FROM e_waste_flags GROUP BY department ORDER BY kg DESC""").fetchall()
    c.close()

    total_co2_saved = sum(r["co2"] for r in reuse)
    total_revenue = sum(r["revenue"] for r in reuse)

    return {
        "total_devices": row["n"],
        "total_kg": round(row["kg"], 1),
        "total_co2_embodied": round(row["co2"], 0),
        "total_co2_saved": round(total_co2_saved, 0),
        "total_revenue_inr": round(total_revenue, 0),
        "by_path": [{"path": REUSE_PATHS[r["recovery_path"]]["label"],
                     "count": r["n"], "co2_saved_kg": round(r["co2"], 1),
                     "revenue_inr": round(r["revenue"], 0)} for r in reuse],
        "by_department": [{"department": d["department"], "devices": d["n"],
                            "kg": round(d["kg"], 1)} for d in by_dept],
    }


flagged, plans = scan_and_plan_e_waste()
summary = e_waste_summary()

print(f"E-waste flagged: {len(flagged)}")
print(f"CO₂ embodied in flagged devices: {summary['total_co2_embodied']:.0f} kg")
print(f"CO₂ recoverable via reuse:      {summary['total_co2_saved']:.0f} kg "
      f"({100*summary['total_co2_saved']/max(summary['total_co2_embodied'],1):.0f}%)")
print(f"Revenue from recycling:         ₹{summary['total_revenue_inr']:.0f}")
print()
print("Recovery path breakdown:")
for p in sorted(summary["by_path"], key=lambda x: -x["co2_saved_kg"]):
    print(f"  {p['path']:32s} {p['count']:3d} devices · {p['co2_saved_kg']:7.0f} kg CO₂ · ₹{p['revenue_inr']:.0f}")
print()
print("By department:")
for d in summary["by_department"]:
    print(f"  {d['department']:8s} {d['devices']:3d} devices · {d['kg']:6.1f} kg recoverable")

E-waste flagged: 90
CO₂ embodied in flagged devices: 28735 kg
CO₂ recoverable via reuse:      17939 kg (62%)
Revenue from recycling:         ₹220700

Recovery path breakdown:
  Refurbish & redeploy              41 devices ·    9240 kg CO₂ · ₹207100
  Donate to school / NGO            34 devices ·    7371 kg CO₂ · ₹0
  Parts harvest for spares          13 devices ·    1118 kg CO₂ · ₹10200
  Certified recycling (CPCB)         2 devices ·     210 kg CO₂ · ₹3400

By department:
  CSE       15 devices ·  222.8 kg recoverable
  MECH      16 devices ·  166.5 kg recoverable
  HUM       16 devices ·  161.8 kg recoverable
  PHY       14 devices ·  146.8 kg recoverable
  ECE       14 devices ·  133.9 kg recoverable
  CIVIL     15 devices ·  107.9 kg recoverable


In [9]:
print(f"ENERGY: {ENERGY.shape}  {ENERGY.index.min()} → {ENERGY.index.max()}")

# Aggregate all buildings into a single daily series
daily_total = ENERGY.sum(axis=1).resample('D').sum()
daily_total = daily_total[daily_total > 0].reset_index()
daily_total.columns = ['date', 'kwh']
daily_total['co2_kg'] = daily_total['kwh'] * 0.71

# Calendar features
daily_total['doy'] = daily_total['date'].dt.dayofyear
daily_total['dow'] = daily_total['date'].dt.dayofweek
daily_total['is_weekend'] = (daily_total['dow'] >= 5).astype(int)

# Lag features
daily_total['lag1'] = daily_total['co2_kg'].shift(1)
daily_total['lag7'] = daily_total['co2_kg'].shift(7)
daily_total['roll7'] = daily_total['co2_kg'].shift(1).rolling(7).mean()

daily = daily_total.dropna().reset_index(drop=True)
print(f"Daily rows: {len(daily)}  avg CO₂ {daily['co2_kg'].mean():.1f} kg/day")

FEATS_CARBON = ['doy', 'dow', 'is_weekend', 'lag1', 'lag7', 'roll7']
split = int(len(daily) * 0.85)
X, y = daily[FEATS_CARBON], daily['co2_kg']

carbon_model = lgb.LGBMRegressor(
    n_estimators=300, learning_rate=0.05, num_leaves=15,
    min_child_samples=5, random_state=42, verbosity=-1)
carbon_model.fit(X.iloc[:split], y.iloc[:split])

pred = carbon_model.predict(X.iloc[split:])
yte = y.iloc[split:]
mae = float(mean_absolute_error(yte, pred))
mape = float((abs(yte - pred) / yte.clip(lower=1)).mean() * 100)

pickle.dump(carbon_model, open(f"{MODEL_DIR}/carbon_forecast.pkl", "wb"))

carbon_stats = {
    "avg_daily_co2_kg": round(float(daily['co2_kg'].mean()), 2),
    "avg_daily_kwh": round(float(daily['kwh'].mean()), 2),
    "mae_kg": round(mae, 3),
    "mape_pct": round(mape, 2),
    "training_days": int(len(daily)),
    "note": "Trained on 5-building aggregate from BDG2 2016-2017 history",
}
json.dump(carbon_stats, open(f"{MODEL_DIR}/carbon_stats.json", "w"), indent=2)

print(f"\nCarbon model: MAE={mae:.2f} kg  MAPE={mape:.2f}%")
print(json.dumps(carbon_stats, indent=2))

ENERGY: (14142, 5)  2016-05-01 19:00:00 → 2017-12-31 23:00:00
Daily rows: 585  avg CO₂ 12080.7 kg/day

Carbon model: MAE=349.41 kg  MAPE=2.96%
{
  "avg_daily_co2_kg": 12080.72,
  "avg_daily_kwh": 17015.1,
  "mae_kg": 349.407,
  "mape_pct": 2.96,
  "training_days": 585,
  "note": "Trained on 5-building aggregate from BDG2 2016-2017 history"
}


In [10]:
# ============================================================
# SIMULATOR CONTROLS
# ============================================================
def toggle_pause():
    STATE["paused"] = not STATE["paused"]
    return "⏸ Paused" if STATE["paused"] else "▶ Running"


def set_speed(x):
    STATE["speed"] = float(x)
    return f"Speed: {STATE['speed']:.1f}s/tick"


def toggle_stream():
    STATE["streaming"] = not STATE["streaming"]
    return "Stream active" if STATE["streaming"] else "Stream paused"


def inject_anomaly():
    c = conn()
    ts = STATE["sim_hour"].isoformat()
    c.execute("UPDATE devices SET state=1")
    c.execute("UPDATE recommendations SET state='REJECTED' WHERE state='PENDING'")
    for r in ROOMS:
        c.execute("INSERT OR REPLACE INTO occupancy VALUES(?,?,?,?,?)",
                  (ts, r["room_id"], 0, "sim", None))
    c.commit(); c.close()
    return f"Anomaly injected · {STATE['sim_hour'].strftime('%H:%M')}"


def reset_sim():
    STATE["sim_hour"] = datetime.now().replace(minute=0, second=0, microsecond=0)
    return f"Reset to {STATE['sim_hour'].strftime('%H:%M')}"


# ============================================================
# YOLO STREAMING
# ============================================================
def run_yolo_stream(frame):
    if frame is None:
        return None, '<div class="ct-empty">Click the webcam box to start.</div>'
    if not STATE["streaming"]:
        return frame, '<div class="ct-empty">Stream paused.</div>'

    h, w = frame.shape[:2]
    scale = min(1.0, 640 / max(h, w, 1))
    frame_in = cv2.resize(frame, (int(w * scale), int(h * scale))) if scale < 1.0 else frame
    r = YOLO_MODEL.predict(frame_in, classes=[0], conf=0.5, verbose=False)[0]
    count = len(r.boxes)
    plotted = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)

    now_ts = time.time()
    if now_ts - STATE["last_occ_write"] > 1.0:
        STATE["last_occ_write"] = now_ts
        try:
            c = conn()
            c.execute("INSERT OR REPLACE INTO occupancy VALUES(?,?,?,?,?)",
                      (datetime.now().isoformat(timespec='seconds'),
                       ROOMS[0]["room_id"], count, "live", None))
            c.commit(); c.close()
        except Exception:
            pass

    badge = f"""
    <div class="ct-metrics">
      <div class="ct-metric" style="border-left-color:#F59E0B">
        <div class="ct-metric-value">{count}</div>
        <div class="ct-metric-label">People detected · live</div>
      </div>
    </div>
    """
    return plotted, badge


# ============================================================
# DATA ACCESS
# ============================================================
def get_live_data():
    c = conn()
    tel = pd.read_sql("SELECT * FROM telemetry ORDER BY ts DESC LIMIT 40", c)
    ghost = pd.read_sql("SELECT * FROM ghost_telemetry ORDER BY ts DESC LIMIT 40", c)
    water = pd.read_sql("SELECT * FROM water_telemetry ORDER BY ts DESC LIMIT 40", c)
    occ = pd.read_sql("SELECT * FROM occupancy WHERE ts=(SELECT MAX(ts) FROM occupancy)", c)
    c.close()
    for df in (tel, ghost, water):
        if not df.empty:
            df['ts'] = pd.to_datetime(df['ts'])
    if not tel.empty: tel = tel.sort_values('ts')
    if not ghost.empty: ghost = ghost.sort_values('ts')
    if not water.empty: water = water.sort_values('ts')
    return tel, ghost, water, occ


def get_decisions():
    c = conn()
    pend = pd.read_sql("SELECT * FROM recommendations WHERE state='PENDING' ORDER BY id DESC LIMIT 10", c)
    refs = pd.read_sql("SELECT * FROM refusals ORDER BY id DESC LIMIT 15", c)
    alerts = pd.read_sql("SELECT * FROM alerts_sent ORDER BY id DESC LIMIT 10", c)
    c.close()
    return pend, refs, alerts


def get_verified():
    c = conn()
    led = pd.read_sql("SELECT * FROM verified_ledger ORDER BY id DESC LIMIT 30", c)
    c.close()
    return led


def _pending_choices():
    """(label, id) pairs for the Decide-tab dropdown, newest first."""
    c = conn()
    rows = c.execute(
        """SELECT id, room_id, device_id, expected_kw FROM recommendations
           WHERE state='PENDING' ORDER BY id DESC"""
    ).fetchall()
    c.close()
    return [(f"#{r['id']} · {r['room_id']} · {r['device_id']} · {r['expected_kw']:.2f} kW", r["id"])
            for r in rows]


def approve_rec(rec_id):
    try: rec_id = int(rec_id)
    except: return "Invalid ID"
    c = conn()
    dev = c.execute("SELECT device_id FROM recommendations WHERE id=?", (rec_id,)).fetchone()
    if not dev:
        c.close(); return f"#{rec_id} not found"
    c.execute("UPDATE devices SET state=0 WHERE id=?", (dev["device_id"],))
    c.execute("""UPDATE recommendations SET state='EXECUTED', executed_at=?
                 WHERE id=? AND state='PENDING'""",
              (datetime.now().isoformat(timespec='seconds'), rec_id))
    c.commit(); c.close()
    return f"Approved #{rec_id}"


def reject_rec(rec_id):
    try: rec_id = int(rec_id)
    except: return "Invalid ID"
    c = conn()
    c.execute("UPDATE recommendations SET state='REJECTED' WHERE id=?", (rec_id,))
    c.commit(); c.close()
    return f"Rejected #{rec_id}"


# ============================================================
# EVENT STREAM
# ============================================================
def get_event_stream(n=25):
    c = conn()
    events = []
    for r in c.execute("""SELECT id, created_at, room_id, device_id, expected_kw
                          FROM recommendations ORDER BY id DESC LIMIT ?""", (n,)):
        events.append({"ts": r["created_at"], "kind": "RECOMMEND",
                       "detail": f"{r['room_id']} · {r['device_id']} · {r['expected_kw']:.2f} kW"})
    for r in c.execute("SELECT ts, room_id, reason FROM refusals ORDER BY id DESC LIMIT ?", (n,)):
        events.append({"ts": r["ts"], "kind": "REFUSAL",
                       "detail": f"{r['room_id']} · {r['reason']}"})
    for r in c.execute("""SELECT created_at, room_id, delta_kw, co2_kg
                          FROM verified_ledger ORDER BY id DESC LIMIT ?""", (n,)):
        events.append({"ts": r["created_at"], "kind": "VERIFIED",
                       "detail": f"{r['room_id']} · {r['delta_kw']:.2f} kW · {r['co2_kg']:.2f} kg"})
    for r in c.execute("""SELECT sent_at, room_id, priority, channel
                          FROM alerts_sent ORDER BY id DESC LIMIT ?""", (n,)):
        events.append({"ts": r["sent_at"], "kind": "ALERT",
                       "detail": f"{r['room_id']} · {r['priority']} via {r['channel']}"})
    c.close()
    events.sort(key=lambda x: x["ts"] or "", reverse=True)
    df = pd.DataFrame(events[:n])
    if not df.empty:
        df.columns = ["Timestamp", "Event", "Detail"]
    return df


# ============================================================
# ADVERSARIAL F1
# ============================================================
def run_adversarial_test():
    room_to_bld = {r["room_id"]: r["building_id"] for r in ROOMS}
    c = conn()
    recs = c.execute("SELECT room_id, created_at FROM recommendations").fetchall()
    c.close()
    recs = [(r["room_id"], r["created_at"]) for r in recs]

    random.seed(11)
    gt = []
    base = datetime.now() - timedelta(days=2)
    for i in range(20):
        r = random.choice(ROOMS)
        start = (base + timedelta(hours=i*2)).replace(minute=0)
        gt.append({"building": r["building_id"], "start": start.isoformat(),
                   "end": (start + timedelta(hours=1)).isoformat()})

    tp = fn = 0
    matched = set()
    for ev in gt:
        for i, (rid, ts) in enumerate(recs):
            if room_to_bld.get(rid) == ev["building"] and ev["start"] <= ts <= ev["end"]:
                if i not in matched:
                    matched.add(i); tp += 1
                break
        else:
            fn += 1

    fp = max(0, len(recs) - len(matched))
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return {"injected_anomalies": len(gt), "recommendations_raised": len(recs),
            "true_positives": tp, "false_positives": fp, "false_negatives": fn,
            "precision": round(prec, 3), "recall": round(rec, 3), "f1": round(f1, 3),
            "ground_truth": "synthetic_injected"}  # disclosed, not external validation


# ============================================================
# MULTI-OBJECTIVE CP-SAT ALLOCATION
# ============================================================
def run_optimizer():
    """
    Multi-objective CP-SAT allocation.
    Hard: one room per session, capacity, room type, no overlap.
    Soft: wasted seats + energy + relocation + peak load + dept affinity.
    """
    random.seed(19)

    dept_of_room = {r["room_id"]: r["dept_id"] for r in ROOMS}
    cap_of_room = {r["room_id"]: r["capacity"] for r in ROOMS}
    type_of_room = {r["room_id"]: r["type"] for r in ROOMS}
    energy_per_room = {r["room_id"]: 0.8 if r["type"] == "lab" else 0.4 for r in ROOMS}

    sessions = []
    for i, (rid, (start, end)) in enumerate(SCHEDULE.items()):
        sessions.append({
            "id": f"S{i+1:02d}", "current_room": rid,
            "dept": dept_of_room[rid],
            "start": start, "duration": end - start,
            "attendance": random.randint(20, min(140, cap_of_room[rid] + 20)),
            "type": type_of_room[rid],
        })

    m = cp_model.CpModel()
    x = {}
    for s in sessions:
        for r in ROOMS:
            if r["type"] != s["type"]: continue
            if r["capacity"] < s["attendance"]: continue
            x[(s["id"], r["room_id"])] = m.NewBoolVar(f"x_{s['id']}_{r['room_id']}")

    for s in sessions:
        vs = [x[(s["id"], r["room_id"])] for r in ROOMS if (s["id"], r["room_id"]) in x]
        if vs:
            m.AddExactlyOne(vs)

    for r in ROOMS:
        rid = r["room_id"]
        for h in range(8, 20):
            overlapping = [x[(s["id"], rid)] for s in sessions
                           if (s["id"], rid) in x and s["start"] <= h < s["start"] + s["duration"]]
            if len(overlapping) > 1:
                m.Add(sum(overlapping) <= 1)

    alpha, beta, gamma, delta, epsilon = 1.0, 3.0, 5.0, 2.0, 1.5
    obj = []
    for s in sessions:
        for r in ROOMS:
            if (s["id"], r["room_id"]) not in x: continue
            var = x[(s["id"], r["room_id"])]
            waste = max(0, r["capacity"] - s["attendance"])
            energy = energy_per_room[r["room_id"]] * s["duration"]
            relocation = 0 if r["room_id"] == s["current_room"] else 1
            dept_mismatch = 0 if r["dept_id"] == s["dept"] else 1
            # FIX: scale consistently with the other three terms below
            # (previously unscaled, so it was ~100-500x underweighted
            # relative to energy/relocation/dept_mismatch — this made
            # "wasted seats", the metric the UI headlines, barely
            # influence the solver despite alpha being set to weight it)
            obj.append(int(alpha * waste * 100) * var)
            obj.append(int(beta * energy * 100) * var)
            obj.append(int(gamma * relocation * 100) * var)
            obj.append(int(epsilon * dept_mismatch * 100) * var)

    for h in range(8, 20):
        active = [x[(s["id"], r["room_id"])] for s in sessions for r in ROOMS
                  if (s["id"], r["room_id"]) in x and s["start"] <= h < s["start"] + s["duration"]]
        if len(active) > 5:
            excess = m.NewIntVar(0, len(active), f"excess_{h}")
            m.Add(excess >= sum(active) - 5)
            obj.append(int(delta * 50) * excess)

    m.Minimize(sum(obj))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 8
    status = solver.Solve(m)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return {"status": solver.StatusName(status), "error": "infeasible"}

    before_waste = sum(max(0, cap_of_room[s["current_room"]] - s["attendance"]) for s in sessions)
    after_waste = 0
    before_energy = sum(energy_per_room[s["current_room"]] * s["duration"] for s in sessions)
    after_energy = 0
    rows = []
    for s in sessions:
        for r in ROOMS:
            if (s["id"], r["room_id"]) in x and solver.Value(x[(s["id"], r["room_id"])]):
                w = max(0, r["capacity"] - s["attendance"])
                e = energy_per_room[r["room_id"]] * s["duration"]
                after_waste += w
                after_energy += e
                rows.append({"Session": s["id"], "Dept": s["dept"],
                             "From": s["current_room"], "To": r["room_id"],
                             "Attendance": s["attendance"], "Capacity": r["capacity"],
                             "Waste": w, "Energy (kWh)": round(e, 2)})

    return {
        "status": solver.StatusName(status),
        "solve_ms": round(solver.WallTime() * 1000, 1),
        "before_waste_seats": before_waste,
        "after_waste_seats": after_waste,
        "waste_reduction_pct": round(100 * (before_waste - after_waste) / max(before_waste, 1), 1),
        "before_energy_kwh": round(before_energy, 2),
        "after_energy_kwh": round(after_energy, 2),
        "energy_reduction_pct": round(100 * (before_energy - after_energy) / max(before_energy, 0.1), 1),
        "assignments": pd.DataFrame(rows),
    }


# ============================================================
# DEPARTMENT SUMMARY
# ============================================================
def get_department_summary():
    c = conn()
    dept_map = {r["room_id"]: r["dept_id"] for r in ROOMS}

    def ensure(d):
        if d not in depts:
            depts[d] = {"department": d, "rooms": 0, "kwh": 0.0, "co2": 0.0,
                        "actions": 0, "pending": 0, "refused": 0,
                        "e_waste_devices": 0, "e_waste_kg": 0.0, "e_waste_co2": 0.0}

    depts = {}
    for r in ROOMS:
        ensure(r["dept_id"]); depts[r["dept_id"]]["rooms"] += 1

    for r in c.execute("""SELECT r.room_id, vl.delta_kw, vl.co2_kg
                          FROM verified_ledger vl JOIN recommendations r ON r.id=vl.rec_id"""):
        d = dept_map.get(r["room_id"], "Unassigned"); ensure(d)
        depts[d]["kwh"] += r["delta_kw"] or 0
        depts[d]["co2"] += r["co2_kg"] or 0
        depts[d]["actions"] += 1

    for r in c.execute("SELECT room_id FROM recommendations WHERE state='PENDING'"):
        d = dept_map.get(r["room_id"], "Unassigned"); ensure(d); depts[d]["pending"] += 1

    for r in c.execute("SELECT room_id FROM refusals"):
        d = dept_map.get(r["room_id"], "Unassigned"); ensure(d); depts[d]["refused"] += 1

    for r in c.execute("""SELECT ef.department, COUNT(*), COALESCE(SUM(ef.kg_recoverable),0),
                          COALESCE(SUM(er.co2_saved_kg),0)
                          FROM e_waste_flags ef
                          LEFT JOIN e_waste_reuse er ON ef.device_id = er.device_id
                          GROUP BY ef.department"""):
        d = r[0]; ensure(d)
        depts[d]["e_waste_devices"] = r[1]
        depts[d]["e_waste_kg"] = round(r[2], 1)
        depts[d]["e_waste_co2"] = round(r[3], 1)

    c.close()
    return depts


def carbon_equivalents(co2_kg):
    return {"trees_years": round(co2_kg / 21.0, 2),
            "km_driven": round(co2_kg / 0.17, 1),
            "led_bulb_hours": int(co2_kg / 0.006),
            "phone_charges": int(co2_kg / 0.0083)}


# ============================================================
# REPORT
# ============================================================
def _generate_executive_summary(identified, total_kwh, total_co2, total_co2_with_reuse):
    """
    Optional LLM enrichment of the report's executive summary paragraph.
    Falls back to the plain template if GEMINI_API_KEY is unset or the
    call fails for ANY reason (network, auth, timeout, malformed reply) —
    report generation must never depend on network access. Every number
    in the template is passed to the model; the prompt explicitly forbids
    inventing new figures, and the report always discloses which path
    produced the summary.
    """
    template = (
        f"The system identified **{identified:.2f} kWh** of avoidable energy use and "
        f"verified **{total_kwh:.2f} kWh** as saved. That is **{total_co2:.2f} kg** of CO₂ "
        f"avoided at the current grid emission factor. Combining verified energy savings "
        f"with e-waste reuse, the total CO₂ impact is **{total_co2_with_reuse:.0f} kg**."
    )
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        return template, "template (no GEMINI_API_KEY set)"
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-1.5-flash")
        prompt = (
            "Rewrite the following campus sustainability report summary in 2-3 "
            "sentences for a university administrator audience. Keep every "
            "number EXACTLY as given below and do not invent, round, or add "
            "any new figures:\n\n" + template
        )
        resp = model.generate_content(prompt, request_options={"timeout": 8})
        text = (resp.text or "").strip()
        return (text, "llm (gemini-1.5-flash)") if text else (template, "template (empty llm reply)")
    except Exception as exc:
        return template, f"template (llm call failed: {type(exc).__name__})"


def ui_report_v2():
    c = conn()
    rows = c.execute("""SELECT r.room_id, vl.delta_kw, vl.co2_kg
                        FROM verified_ledger vl JOIN recommendations r ON r.id=vl.rec_id""").fetchall()
    dept_map = {r["room_id"]: r["dept_id"] for r in ROOMS}
    depts = {}
    for row in rows:
        d = dept_map.get(row["room_id"], "Unassigned")
        depts.setdefault(d, {"kwh": 0.0, "co2": 0.0, "actions": 0})
        depts[d]["kwh"] += row["delta_kw"] or 0
        depts[d]["co2"] += row["co2_kg"] or 0
        depts[d]["actions"] += 1

    identified = c.execute("""SELECT COALESCE(SUM(expected_kw),0) FROM recommendations
                              WHERE state IN ('PENDING','APPROVED')""").fetchone()[0]
    refusals = {r["reason"]: r["n"] for r in c.execute(
        "SELECT reason, COUNT(*) as n FROM refusals GROUP BY reason").fetchall()}
    proposed = c.execute("SELECT COUNT(*) FROM recommendations").fetchone()[0]
    verified = c.execute("SELECT COUNT(*) FROM recommendations WHERE state='VERIFIED'").fetchone()[0]
    c.close()

    ew = e_waste_summary()
    try: carbon_stats = json.load(open(f"{MODEL_DIR}/carbon_stats.json"))
    except: carbon_stats = {}

    total_kwh = sum(x["kwh"] for x in depts.values())
    total_co2 = sum(x["co2"] for x in depts.values())
    total_co2_with_reuse = total_co2 + ew["total_co2_saved"]

    summary_text, summary_source = _generate_executive_summary(
        identified, total_kwh, total_co2, total_co2_with_reuse)

    lines = [
        "# Campus Twin AI — Session Report", "",
        f"**Generated:** {datetime.now().isoformat(timespec='seconds')}  ",
        f"**Grid emission factor:** 0.71 kgCO₂/kWh (CEA baseline)", "",
        "## Executive summary", "",
        summary_text, "",
        f"_Summary source: {summary_source}_", "",
        "## Energy totals", "", "| Metric | Value |", "|---|---|",
        f"| Identified avoidable energy | {identified:.2f} kWh |",
        f"| Verified savings | {total_kwh:.2f} kWh |",
        f"| Verified CO₂ avoided | {total_co2:.2f} kg |",
        f"| E-waste reuse CO₂ avoided | {ew['total_co2_saved']:.0f} kg |",
        f"| **Total CO₂ impact** | **{total_co2_with_reuse:.0f} kg** |",
        f"| Recommendations raised | {proposed} |",
        f"| Recommendations verified | {verified} |",
        f"| Refusals logged | {sum(refusals.values())} |", "",
        "## Department accountability", "",
        "| Department | Verified kWh | CO₂ (kg) | Actions | E-waste devices | E-waste CO₂ (kg) |",
        "|---|---|---|---|---|---|",
    ]
    for d, v in sorted(depts.items(), key=lambda x: -x[1]["kwh"]):
        lines.append(f"| {d} | {v['kwh']:.2f} | {v['co2']:.2f} | {v['actions']} | "
                     f"{v['e_waste_devices']} | {v['e_waste_co2']:.0f} |")
    lines.append("")

    if refusals:
        lines += ["## Why actions were refused", "", "| Reason | Count |", "|---|---|"]
        for r, n in sorted(refusals.items(), key=lambda x: -x[1]):
            lines.append(f"| {r.replace('_', ' ')} | {n} |")
        lines.append("")

    lines += ["## E-waste reuse plan", "",
              f"**Devices flagged:** {ew['total_devices']}  ",
              f"**CO₂ avoided via reuse:** {ew['total_co2_saved']:.0f} kg  ",
              f"**Revenue from recycling:** ₹{ew['total_revenue_inr']:.0f}", "",
              "| Recovery path | Devices | CO₂ avoided (kg) | Revenue (₹) |",
              "|---|---|---|---|"]
    for p in ew["by_path"]:
        lines.append(f"| {p['path']} | {p['count']} | {p['co2_saved_kg']:.0f} | {p['revenue_inr']:.0f} |")
    lines.append("")

    if carbon_stats:
        lines += ["## Carbon baseline", "",
                  f"- Average daily CO₂: **{carbon_stats.get('avg_daily_co2_kg', 0):.2f} kg**",
                  f"- Forecast accuracy: **MAPE {carbon_stats.get('mape_pct', 0):.1f}%**", ""]

    lines += ["## Method", "",
              "Energy savings use a **counterfactual ghost twin**. E-waste CO₂ uses "
              "embodied carbon recovery percentages from reuse path (refurbish 70%, donate 65%, "
              "parts 40%, recycle 15%).", "",
              "---", "", "_Generated by Campus Twin AI. No IoT sensors were used._"]

    md = "\n".join(lines)
    ts = datetime.now().strftime("%H%M%S")
    md_path = f"/kaggle/working/report_{ts}.md"
    json_path = f"/kaggle/working/report_{ts}.json"
    with open(md_path, "w") as f: f.write(md)
    with open(json_path, "w") as f:
        json.dump({"markdown": md, "departments": depts, "e_waste": ew}, f, indent=2)
    return md, md_path, json_path


print("Helpers loaded.")
print(f"  Scheduled sessions: {len(SCHEDULE)}")
print(f"  Buildings: {len(ROOMS)} rooms across {len(BUILDINGS)} buildings")
print(f"  Departments: {len(DEPTS)}")

Helpers loaded.
  Scheduled sessions: 20
  Buildings: 30 rooms across 5 buildings
  Departments: 6


In [11]:
# ==========================================================================
# CAMPUS TWIN AI — GRADIO UI 


import traceback

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500;600;700&family=Space+Grotesk:wght@400;500;600;700&family=Inter:wght@400;500;600;700&display=swap');

:root{
  --bg:#0B1220; --surface:#111A2C; --surface-2:#182238; --surface-3:#1F2A44;
  --border:#243352; --border-2:#324369;
  --ink:#FFFFFF; --text:#E6EDF7; --text-2:#B6C2D6; --muted:#8794AB; --dim:#5E6B85;
  --emerald:#10B981; --emerald-bright:#34D399; --emerald-dim:rgba(16,185,129,0.15); --emerald-soft:#052E22;
  --amber:#F59E0B; --amber-bright:#FBBF24; --amber-dim:rgba(245,158,11,0.15); --amber-soft:#3B2409;
  --red:#EF4444; --red-bright:#F87171; --red-dim:rgba(239,68,68,0.15); --red-soft:#3B0F0F;
  --blue:#3B82F6; --blue-bright:#60A5FA; --blue-dim:rgba(59,130,246,0.15); --blue-soft:#0A1E3D;
  --violet:#8B5CF6;
}

html, body, .gradio-container, gradio-app {
  background-color: var(--bg) !important;
  color: var(--text) !important;
  font-family: 'Inter', sans-serif !important;
}
.gradio-container{max-width:1560px!important;margin:0 auto!important;padding:0 24px!important;}

.gradio-container, .gradio-container * {
  --body-background-fill: var(--bg) !important;
  --background-fill-primary: var(--surface) !important;
  --background-fill-secondary: var(--surface-2) !important;
  --border-color-primary: var(--border) !important;
  --border-color-accent: var(--border-2) !important;
  --block-background-fill: var(--surface) !important;
  --block-border-color: var(--border) !important;
  --input-background-fill: var(--surface-2) !important;
  --input-border-color: var(--border-2) !important;
  --color-accent: var(--emerald) !important;
  --color-accent-soft: var(--emerald-dim) !important;
  --neutral-50: var(--surface) !important;
  --neutral-100: var(--surface-2) !important;
  --neutral-200: var(--border) !important;
  --neutral-300: var(--border-2) !important;
  --neutral-400: var(--dim) !important;
  --neutral-500: var(--muted) !important;
  --neutral-600: var(--text-2) !important;
  --neutral-700: var(--text) !important;
  --neutral-800: var(--text) !important;
  --neutral-900: var(--ink) !important;
}

/* ---------- Header ---------- */
.ct-header{
  display:flex;align-items:center;justify-content:space-between;
  padding:26px 34px;
  background:linear-gradient(120deg, #111A2C 0%, #162440 55%, #1A2E48 100%);
  border:1px solid var(--border-2);border-radius:14px;
  margin:20px 0 24px;box-shadow:0 8px 32px rgba(0,0,0,0.35);
}
.ct-brand{display:flex;align-items:center;gap:18px;}
.ct-logo{
  width:56px;height:56px;border-radius:14px;
  background:linear-gradient(135deg, #10B981, #059669);
  color:#FFFFFF;display:flex;align-items:center;justify-content:center;
  font-family:'Space Grotesk',sans-serif;font-size:28px;font-weight:700;
  box-shadow:0 8px 24px rgba(16,185,129,0.45);
}
.ct-title{
  font-family:'Space Grotesk',sans-serif;font-size:28px;font-weight:700;
  color:#FFFFFF;letter-spacing:-0.02em;line-height:1.1;
}
.ct-subtitle{
  font-family:'JetBrains Mono',monospace;font-size:12px;
  color:var(--text-2);margin-top:8px;letter-spacing:0.05em;font-weight:500;
}
.ct-status{
  display:flex;align-items:center;gap:10px;padding:12px 20px;
  background:var(--emerald-dim);border:1px solid rgba(16,185,129,0.4);
  border-radius:8px;font-family:'JetBrains Mono',monospace;font-size:11.5px;
  font-weight:700;color:var(--emerald-bright);letter-spacing:0.08em;
}
.ct-status-dot{
  width:8px;height:8px;border-radius:50%;background:var(--emerald-bright);
  animation:pulse 1.8s infinite;box-shadow:0 0 12px var(--emerald-bright);
}
@keyframes pulse{0%,100%{opacity:1;transform:scale(1)}50%{opacity:0.5;transform:scale(0.85)}}

/* ---------- Tabs ---------- */
.tabs{background:transparent!important;border-bottom:1px solid var(--border-2)!important;}
.tabs button{
  font-family:'JetBrains Mono',monospace!important;
  font-weight:600!important;font-size:12px!important;
  color:var(--muted)!important;
  text-transform:uppercase!important;letter-spacing:0.1em!important;
  border:none!important;border-bottom:3px solid transparent!important;
  background:transparent!important;padding:18px 24px!important;
  border-radius:0!important;transition:all 0.15s ease!important;
}
.tabs button:hover{color:var(--emerald-bright)!important;background:var(--emerald-soft)!important;}
.tabs button.selected{
  color:var(--emerald-bright)!important;
  border-bottom-color:var(--emerald-bright)!important;
  background:var(--emerald-soft)!important;
}

/* ---------- Tab Intro Card ---------- */
.ct-tab-intro{
  background:var(--surface);border:1px solid var(--border-2);
  border-left:4px solid var(--emerald);border-radius:10px;
  padding:18px 24px;margin:20px 0;box-shadow:0 4px 16px rgba(0,0,0,0.2);
}
.ct-tab-intro-title{
  font-family:'Space Grotesk',sans-serif;font-size:16px;
  font-weight:700;color:#FFFFFF;margin-bottom:8px;letter-spacing:-0.01em;
}
.ct-tab-intro-text{font-size:13.5px;color:var(--text-2);line-height:1.65;}
.ct-tab-intro-text code{
  font-family:'JetBrains Mono',monospace;font-size:12.5px;
  background:var(--surface-3);padding:3px 8px;border-radius:4px;
  color:var(--emerald-bright);font-weight:600;border:1px solid var(--border-2);
}

/* ---------- Caveat ---------- */
.ct-caveat{
  background:var(--amber-soft);border:1px solid rgba(245,158,11,0.4);
  border-radius:8px;padding:12px 16px;margin-bottom:14px;
  font-size:13px;color:var(--amber-bright);line-height:1.55;font-weight:500;
}

/* ---------- Metric Cards ---------- */
.ct-metrics{
  display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));
  gap:16px;margin:20px 0;
}
.ct-metric{
  background:linear-gradient(160deg, var(--surface) 0%, var(--surface-2) 100%);
  border:1px solid var(--border-2);border-radius:12px;
  padding:20px 24px 18px;border-left:4px solid var(--emerald);
  position:relative;overflow:hidden;transition:all 0.2s ease;
  box-shadow:0 4px 14px rgba(0,0,0,0.2);
}
.ct-metric:hover{border-color:var(--border-2);transform:translateY(-2px);
                 box-shadow:0 8px 24px rgba(0,0,0,0.35);}
.ct-metric::before{
  content:'';position:absolute;top:0;right:0;width:90px;height:90px;
  background:radial-gradient(circle at top right,rgba(16,185,129,0.18),transparent 70%);
  pointer-events:none;
}
.ct-metric.warn{border-left-color:var(--amber-bright);background:linear-gradient(160deg, var(--surface) 0%, var(--amber-soft) 130%);}
.ct-metric.alert{border-left-color:var(--red-bright);background:linear-gradient(160deg, var(--surface) 0%, var(--red-soft) 130%);}
.ct-metric.success{border-left-color:var(--emerald-bright);background:linear-gradient(160deg, var(--surface) 0%, var(--emerald-soft) 130%);}
.ct-metric.info{border-left-color:var(--blue-bright);background:linear-gradient(160deg, var(--surface) 0%, var(--blue-soft) 130%);}
.ct-metric.muted{border-left-color:var(--dim);}

.ct-metric-value{
  font-family:'JetBrains Mono',monospace;font-size:30px;font-weight:700;
  color:#FFFFFF;letter-spacing:-0.03em;line-height:1.05;
  font-variant-numeric:tabular-nums;text-shadow:0 2px 12px rgba(0,0,0,0.3);
}
.ct-metric.success .ct-metric-value{color:var(--emerald-bright);}
.ct-metric.warn .ct-metric-value{color:var(--amber-bright);}
.ct-metric.alert .ct-metric-value{color:var(--red-bright);}
.ct-metric.info .ct-metric-value{color:var(--blue-bright);}
.ct-metric-label{
  font-family:'JetBrains Mono',monospace;font-size:11px;
  color:var(--text-2);margin-top:12px;font-weight:600;
  text-transform:uppercase;letter-spacing:0.1em;
}
.ct-metric-help{
  font-size:11.5px;color:var(--muted);margin-top:6px;
  font-family:'Inter',sans-serif;letter-spacing:0;
  text-transform:none;font-weight:400;font-style:italic;line-height:1.5;
}
.ct-metric-unit{
  font-family:'JetBrains Mono',monospace;font-size:15px;
  color:var(--text-2);font-weight:500;margin-left:5px;
}

/* ---------- Inline status pill ---------- */
.ct-inline-status{
  display:inline-block;font-family:'JetBrains Mono',monospace;
  font-size:11.5px;color:var(--emerald-bright);
  padding:7px 14px;background:var(--emerald-soft);
  border:1px solid rgba(16,185,129,0.35);border-radius:8px;
  letter-spacing:0.03em;font-weight:600;margin:4px 0;
}
.ct-inline-status.muted{
  color:var(--muted);background:var(--surface-2);border-color:var(--border-2);
}
.ct-inline-status.warn{
  color:var(--amber-bright);background:var(--amber-soft);
  border-color:rgba(245,158,11,0.4);
}

/* ---------- Buttons ---------- */
button.primary, button.lg.primary, button.sm.primary{
  background:linear-gradient(135deg, #10B981, #059669)!important;
  border:1px solid var(--emerald-bright)!important;
  color:#FFFFFF!important;font-family:'JetBrains Mono',monospace!important;
  font-weight:700!important;font-size:12.5px!important;
  text-transform:uppercase!important;letter-spacing:0.06em!important;
  padding:13px 26px!important;border-radius:8px!important;
  transition:all 0.15s ease!important;
  box-shadow:0 4px 14px rgba(16,185,129,0.35)!important;
}
button.primary:hover{
  background:linear-gradient(135deg, #34D399, #10B981)!important;
  box-shadow:0 6px 22px rgba(16,185,129,0.55)!important;
  transform:translateY(-1px);
}
button.secondary, button.lg.secondary, button.sm.secondary{
  background:var(--surface-2)!important;
  border:1px solid var(--border-2)!important;
  color:var(--text)!important;font-family:'JetBrains Mono',monospace!important;
  font-size:12.5px!important;font-weight:600!important;
  text-transform:uppercase!important;letter-spacing:0.05em!important;
  padding:12px 22px!important;border-radius:8px!important;
  transition:all 0.15s ease!important;
}
button.secondary:hover{
  border-color:var(--emerald-bright)!important;
  color:var(--emerald-bright)!important;
  background:var(--emerald-soft)!important;
}

/* ---------- Dataframe ---------- */
.dataframe{
  font-family:'JetBrains Mono',monospace!important;font-size:12.5px!important;
  background:var(--surface)!important;border:1px solid var(--border-2)!important;
  border-radius:10px!important;overflow:hidden!important;
}
.dataframe table{background:transparent!important;width:100%!important;border-collapse:collapse!important;}
.dataframe thead tr{background:var(--surface-3)!important;}
.dataframe th{
  background:var(--surface-3)!important;color:#FFFFFF!important;
  font-family:'JetBrains Mono',monospace!important;
  font-weight:700!important;font-size:11px!important;
  text-transform:uppercase!important;letter-spacing:0.08em!important;
  padding:14px 16px!important;border-bottom:1px solid var(--border-2)!important;
}
.dataframe td{
  background:var(--surface)!important;color:var(--text)!important;
  padding:12px 16px!important;border-bottom:1px solid var(--border)!important;
  font-family:'JetBrains Mono',monospace!important;font-size:12.5px!important;
}
.dataframe tbody tr:hover td{background:var(--surface-2)!important;}

/* ---------- Inputs ---------- */
input[type="number"], input[type="text"], textarea, select, .gr-text-input{
  font-family:'JetBrains Mono',monospace!important;
  background:var(--surface-2)!important;
  border:1px solid var(--border-2)!important;
  border-radius:8px!important;padding:11px 15px!important;
  font-size:13px!important;color:#FFFFFF!important;
  transition:all 0.15s ease!important;
}
input[type="number"]:focus, input[type="text"]:focus,
textarea:focus, select:focus{
  border-color:var(--emerald-bright)!important;
  box-shadow:0 0 0 3px var(--emerald-dim)!important;
  outline:none!important;
}
label{
  color:var(--text-2)!important;
  font-family:'JetBrains Mono',monospace!important;
  font-size:11.5px!important;text-transform:uppercase!important;
  letter-spacing:0.05em!important;font-weight:600!important;
}
.gr-dropdown input, .svelte-1ipelgc{
  background:var(--surface-2)!important;color:#FFFFFF!important;
  border-color:var(--border-2)!important;
}
ul.options, .options{
  background:var(--surface-2)!important;border:1px solid var(--border-2)!important;
  color:#FFFFFF!important;
}
ul.options li:hover{
  background:var(--emerald-soft)!important;color:var(--emerald-bright)!important;
}

/* ============================================================
   ---------- SLIDER FIX ----------
   ============================================================ */
.gradio-container .block:has(input[type="range"]) {
  display: flex !important;
  flex-direction: column !important;
  align-items: stretch !important;
  gap: 0 !important;
  overflow: visible !important;
  padding: 14px 16px !important;
}
.gradio-container .block:has(input[type="range"]) .label-wrap,
.gradio-container .block:has(input[type="range"]) > .head,
.gradio-container .block:has(input[type="range"]) > div:has(> [data-testid="block-label"]),
.gradio-container .block:has(input[type="range"]) > div:has(> span[data-testid="block-label"]) {
  display: flex !important;
  flex-direction: column !important;
  align-items: flex-start !important;
  flex-wrap: nowrap !important;
  gap: 0 !important;
  width: 100% !important;
  height: auto !important;
  min-height: 0 !important;
  max-height: none !important;
  position: static !important;
  margin: 0 0 8px 0 !important;
  padding: 0 !important;
  overflow: visible !important;
}
.gradio-container .block:has(input[type="range"]) [data-testid="block-label"],
.gradio-container .block:has(input[type="range"]) label > span:first-child {
  display: block !important;
  position: static !important;
  width: auto !important;
  max-width: 100% !important;
  margin: 0 !important;
  padding: 0 !important;
  line-height: 1.4 !important;
  white-space: normal !important;
  overflow: visible !important;
  text-overflow: clip !important;
}
.gradio-container .block:has(input[type="range"]) [data-testid="block-info"],
.gradio-container .block:has(input[type="range"]) .info,
.gradio-container [data-testid="block-info"] {
  display: block !important;
  position: static !important;
  width: auto !important;
  max-width: 100% !important;
  margin: 3px 0 0 0 !important;
  padding: 0 !important;
  line-height: 1.4 !important;
  white-space: normal !important;
  overflow: visible !important;
  text-overflow: clip !important;
  text-transform: none !important;
  font-style: italic !important;
  font-weight: 400 !important;
  font-size: 11px !important;
  letter-spacing: 0.02em !important;
  color: var(--muted) !important;
  font-family: 'JetBrains Mono', monospace !important;
}
.gradio-container label [data-testid="block-info"],
.gradio-container label span.info { text-transform: none !important; }
.gradio-container .block:has(input[type="range"]) .slider-row,
.gradio-container .block:has(input[type="range"]) > div:has(> input[type="range"]) {
  display: flex !important;
  flex-direction: row !important;
  flex-wrap: nowrap !important;
  align-items: center !important;
  gap: 12px !important;
  width: 100% !important;
  margin: 0 !important;
  padding: 0 !important;
}
.gradio-container input[type="range"]{
  flex: 1 1 auto !important;
  width: 100% !important;
  min-width: 0 !important;
  height: 6px !important;
  margin: 6px 0 2px 0 !important;
  background: var(--border-2) !important;
  border-radius: 3px !important;
  appearance: none !important;
  -webkit-appearance: none !important;
}
.gradio-container input[type="range"]::-webkit-slider-thumb{
  -webkit-appearance: none !important;
  appearance: none !important;
  width: 16px !important;
  height: 16px !important;
  border-radius: 50% !important;
  background: var(--emerald-bright) !important;
  box-shadow: 0 0 10px var(--emerald-bright) !important;
  cursor: pointer !important;
}
.gradio-container input[type="range"]::-moz-range-thumb{
  width: 16px !important;
  height: 16px !important;
  border: none !important;
  border-radius: 50% !important;
  background: var(--emerald-bright) !important;
  box-shadow: 0 0 10px var(--emerald-bright) !important;
  cursor: pointer !important;
}
.gradio-container .block:has(input[type="range"]) input[type="number"],
.gradio-container .block:has(input[type="range"]) .slider_input_container textarea {
  display: none !important;
}

/* ---------- Markdown / Report ---------- */
.prose, .prose * {background-color:transparent !important;}
.prose{
  background:var(--surface)!important;border:1px solid var(--border-2)!important;
  border-radius:10px!important;padding:30px 38px!important;
  box-shadow:0 4px 16px rgba(0,0,0,0.2)!important;
}
.prose h1{
  font-family:'Space Grotesk',sans-serif!important;font-size:32px!important;
  color:#FFFFFF!important;font-weight:700!important;
  margin-bottom:18px!important;letter-spacing:-0.02em;
}
.prose h2{
  font-family:'Space Grotesk',sans-serif!important;font-size:20px!important;
  color:var(--emerald-bright)!important;margin:28px 0 14px!important;
  border-bottom:1px solid var(--border-2)!important;
  padding-bottom:12px!important;font-weight:700!important;
}
.prose h3{
  font-family:'Space Grotesk',sans-serif!important;font-size:16px!important;
  color:#FFFFFF!important;margin:20px 0 10px!important;font-weight:700;
}
.prose p,.prose li{
  color:var(--text)!important;font-size:14px!important;line-height:1.7!important;
}
.prose strong{color:var(--emerald-bright)!important;font-weight:700;}
.prose em{color:var(--text-2)!important;}
.prose table{width:100%!important;border-collapse:collapse!important;margin:16px 0!important;}
.prose table th{
  text-align:left!important;padding:12px 16px!important;
  background:var(--surface-3)!important;color:#FFFFFF!important;
  font-family:'JetBrains Mono',monospace!important;
  font-size:11px!important;text-transform:uppercase!important;
  letter-spacing:0.08em!important;font-weight:700!important;
  border-bottom:1px solid var(--border-2)!important;
}
.prose table td{
  padding:12px 16px!important;border-bottom:1px solid var(--border)!important;
  font-family:'JetBrains Mono',monospace!important;
  font-size:12.5px!important;color:var(--text)!important;
}
.prose code{
  background:var(--surface-3)!important;padding:3px 8px!important;
  border-radius:4px!important;
  font-family:'JetBrains Mono',monospace!important;
  font-size:12.5px!important;color:var(--emerald-bright)!important;
  font-weight:600!important;border:1px solid var(--border-2);
}
.prose a{color:var(--blue-bright)!important;}

/* ---------- Empty States ---------- */
.ct-empty{
  padding:52px 28px;text-align:center;color:var(--text-2);
  font-family:'JetBrains Mono',monospace;font-size:13px;
  background:var(--surface);border:1px dashed var(--border-2);
  border-radius:12px;line-height:1.7;
}
.ct-empty strong{color:#FFFFFF;font-weight:700;display:block;font-size:14px;margin-bottom:6px;}
.ct-empty-hint{
  font-size:12px;color:var(--muted);margin-top:10px;
  font-family:'Inter',sans-serif;font-style:italic;
}

/* ---------- Section Labels ---------- */
.ct-section-label{
  font-family:'JetBrains Mono',monospace;font-size:12px;font-weight:700;
  color:var(--text-2);letter-spacing:0.14em;text-transform:uppercase;
  margin:28px 0 18px;display:flex;align-items:center;gap:14px;
}
.ct-section-label::after{
  content:'';flex:1;height:1px;
  background:linear-gradient(90deg,var(--border-2) 0%,transparent 100%);
}
.ct-section-label .hint{
  font-family:'Inter',sans-serif;font-size:12px;color:var(--muted);
  text-transform:none;letter-spacing:0;font-weight:400;font-style:italic;
}

/* ---------- Footer ---------- */
.ct-footer{
  margin-top:52px;padding:24px 0;
  border-top:1px solid var(--border-2);
  display:flex;justify-content:space-between;flex-wrap:wrap;gap:12px;
  color:var(--muted);font-family:'JetBrains Mono',monospace;
  font-size:11px;text-transform:uppercase;letter-spacing:0.1em;font-weight:500;
}

/* ---------- Plotly ---------- */
.js-plotly-plot{
  border:1px solid var(--border-2)!important;border-radius:12px!important;
  background:var(--surface)!important;box-shadow:0 4px 16px rgba(0,0,0,0.2)!important;
  overflow:hidden;
}
.js-plotly-plot .plotly .modebar{background:transparent!important;}
.js-plotly-plot .plotly .modebar-btn path{fill:var(--text-2)!important;}
.js-plotly-plot .plotly .modebar-btn:hover path{fill:var(--emerald-bright)!important;}
.gradio-container .js-plotly-plot .plotly text{
  fill:#E6EDF7 !important;font-family:'JetBrains Mono',monospace !important;
}

/* ---------- Event Stream ---------- */
.ct-event-list{
  background:var(--surface);border:1px solid var(--border-2);
  border-radius:12px;overflow:hidden;box-shadow:0 4px 16px rgba(0,0,0,0.2);
}
.ct-event{
  display:grid;grid-template-columns:80px 110px 1fr;gap:16px;align-items:center;
  padding:12px 18px;border-bottom:1px solid var(--border);
  font-family:'JetBrains Mono',monospace;font-size:12.5px;
}
.ct-event:last-child{border-bottom:none;}
.ct-event:hover{background:var(--surface-2);}
.ct-event-time{color:var(--muted);font-size:11.5px;font-weight:500;}
.ct-event-kind{
  font-size:10.5px;font-weight:700;padding:5px 10px;border-radius:5px;
  text-align:center;letter-spacing:0.05em;text-transform:uppercase;
}
.kind-RECOMMEND{color:var(--emerald-bright);background:var(--emerald-dim);border:1px solid rgba(16,185,129,0.4);}
.kind-REFUSAL{color:var(--amber-bright);background:var(--amber-dim);border:1px solid rgba(245,158,11,0.4);}
.kind-VERIFIED{color:var(--emerald-bright);background:var(--emerald-dim);border:1px solid rgba(16,185,129,0.5);}
.kind-ALERT{color:var(--red-bright);background:var(--red-dim);border:1px solid rgba(239,68,68,0.4);}
.ct-event-detail{color:var(--text);}

/* ---------- Legend Row ---------- */
.ct-legend{
  display:flex;flex-wrap:wrap;gap:20px;margin:16px 0 22px;
  padding:14px 20px;background:var(--surface);
  border:1px solid var(--border-2);border-radius:10px;
  font-family:'JetBrains Mono',monospace;font-size:12px;
  box-shadow:0 2px 10px rgba(0,0,0,0.15);
}
.ct-legend-item{display:flex;align-items:center;gap:10px;color:var(--text-2);}
.ct-legend-swatch{
  width:12px;height:12px;border-radius:3px;flex-shrink:0;
  box-shadow:0 0 8px currentColor;
}
.ct-legend-label{color:#FFFFFF;font-weight:600;letter-spacing:0.03em;}

/* ---------- Neutralize remaining light-mode wrappers ---------- */
.gradio-container .block,
.gradio-container .form,
.gradio-container .panel,
.gradio-container fieldset,
.gradio-container .gr-group,
.gradio-container .gr-box {
  background-color: transparent !important;
  border-color: var(--border) !important;
}
"""

PLOT_LAYOUT = dict(
    plot_bgcolor="#111A2C", paper_bgcolor="#111A2C",
    font=dict(family="JetBrains Mono, monospace", size=12, color="#E6EDF7"),
    margin=dict(t=60, b=55, l=65, r=30),
    xaxis=dict(gridcolor="#243352", zerolinecolor="#324369",
               title_font=dict(size=12, color="#B6C2D6"),
               tickfont=dict(size=11, color="#B6C2D6"),
               linecolor="#324369"),
    yaxis=dict(gridcolor="#243352", zerolinecolor="#324369",
               title_font=dict(size=12, color="#B6C2D6"),
               tickfont=dict(size=11, color="#B6C2D6"),
               linecolor="#324369"),
    legend=dict(orientation="h", y=-0.25,
                bgcolor="rgba(17,26,44,0.6)",
                bordercolor="#324369", borderwidth=1,
                font=dict(size=11, color="#E6EDF7")),
    hoverlabel=dict(bgcolor="#1F2A44", bordercolor="#324369",
                    font=dict(family="JetBrains Mono", size=12, color="#FFFFFF")),
)

HEADER_HTML = f"""
<div class="ct-header">
  <div class="ct-brand">
    <div class="ct-logo">◈</div>
    <div>
      <div class="ct-title">Campus Twin AI</div>
      <div class="ct-subtitle">DUAL-TWIN COUNTERFACTUAL ENGINE · ZERO IoT SENSORS · LOCAL-ONLY</div>
    </div>
  </div>
  <div class="ct-status">
    <span class="ct-status-dot"></span>
    <span>LIVE · {datetime.now().strftime('%H:%M:%S')}</span>
  </div>
</div>
"""

FOOTER_HTML = """
<div class="ct-footer">
  <span>Campus Twin AI · counterfactual verification · no IoT hardware</span>
  <span>LightGBM forecasting · CP-SAT allocation · e-waste reuse</span>
</div>
"""

# ==========================================================================
# HTML HELPERS
# ==========================================================================
def _inline_status(text, style=""):
    cls = f"ct-inline-status {style}".strip()
    return f'<div class="{cls}">{text}</div>'

def _metric_card(value, label, help_text="", style="", unit=""):
    unit_html = f'<span class="ct-metric-unit">{unit}</span>' if unit else ""
    help_html = f'<div class="ct-metric-help">{help_text}</div>' if help_text else ""
    return f"""
    <div class="ct-metric {style}">
      <div class="ct-metric-value">{value}{unit_html}</div>
      <div class="ct-metric-label">{label}</div>
      {help_html}
    </div>
    """

def _metrics_html(pending=0, verified=0, co2=0.0, alerts=0, sim_hour=""):
    return f"""
    <div class="ct-metrics">
      {_metric_card(sim_hour, "Simulated Hour", "Current time in the accelerated twin", "info")}
      {_metric_card(pending, "Pending Approvals", "Awaiting human decision", "warn")}
      {_metric_card(verified, "Verified Actions", "Executed and counterfactually confirmed", "success")}
      {_metric_card(f"{co2:.2f}", "CO₂ Avoided", "Measured, not estimated", "success", "kg")}
      {_metric_card(alerts, "Alerts Dispatched", "Sent to admin via Telegram", "alert")}
    </div>
    """

def _empty_state(title, hint=""):
    hint_html = f'<div class="ct-empty-hint">{hint}</div>' if hint else ""
    return f'<div class="ct-empty"><strong>{title}</strong>{hint_html}</div>'

def _tab_intro(title, text):
    return f"""
    <div class="ct-tab-intro">
      <div class="ct-tab-intro-title">{title}</div>
      <div class="ct-tab-intro-text">{text}</div>
    </div>
    """

def _legend(items):
    swatches = "".join(
        f'<div class="ct-legend-item">'
        f'<span class="ct-legend-swatch" style="background:{color};color:{color}"></span>'
        f'<span class="ct-legend-label">{label}</span>'
        f'</div>'
        for label, color in items
    )
    return f'<div class="ct-legend">{swatches}</div>'

def _empty_df(cols):
    return pd.DataFrame(columns=cols)

def _placeholder_fig(title, message="Awaiting data · click the refresh button to load"):
    fig = go.Figure()
    fig.add_annotation(
        text=message, xref="paper", yref="paper", x=0.5, y=0.5,
        showarrow=False, font=dict(size=13, color="#8794AB",
                                   family="JetBrains Mono, monospace"))
    fig.update_layout(
        height=340,
        title=dict(text=title, font=dict(size=15, color="#FFFFFF")),
        **PLOT_LAYOUT)
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig

# ==========================================================================
# UI DATA FUNCTIONS
# ==========================================================================
def ui_metrics():
    try:
        c = conn()
        p = c.execute("SELECT COUNT(*) FROM recommendations WHERE state='PENDING'").fetchone()[0]
        v = c.execute("SELECT COUNT(*) FROM verified_ledger").fetchone()[0]
        co2 = c.execute("SELECT COALESCE(SUM(co2_kg),0) FROM verified_ledger").fetchone()[0]
        a = c.execute("SELECT COUNT(*) FROM alerts_sent").fetchone()[0]
        c.close()
        return _metrics_html(p, v, co2, a, STATE['sim_hour'].strftime('%d %b %H:%M'))
    except Exception:
        return _metrics_html(0, 0, 0, 0, "—")

def ui_refresh_live(img):
    try:
        _, badge = run_yolo_stream(img)
    except Exception:
        badge = _empty_state("Camera idle", "Click the webcam box to start detection")

    try:
        tel, ghost, water, occ = get_live_data()
    except Exception:
        tel = ghost = water = occ = pd.DataFrame()

    colors = ["#10B981", "#3B82F6", "#8B5CF6", "#F59E0B", "#EF4444"]

    tel_fig = go.Figure()
    if not tel.empty and 'building' in tel.columns and 'kw' in tel.columns:
        for i, b in enumerate(tel.building.unique()):
            d = tel[tel.building == b]
            tel_fig.add_trace(go.Scatter(
                x=d['ts'], y=d['kw'], name=f"Building {b}",
                mode='lines+markers',
                line=dict(color=colors[i % len(colors)], width=2.5),
                marker=dict(size=6, line=dict(color="#111A2C", width=1)),
                hovertemplate="<b>%{fullData.name}</b><br>"
                              "Time: %{x|%H:%M}<br>"
                              "Power: %{y:.2f} kW<extra></extra>"))
    else:
        tel_fig.add_annotation(text="No telemetry yet — simulator warming up",
                                xref="paper", yref="paper", x=0.5, y=0.5,
                                showarrow=False,
                                font=dict(size=13, color="#8794AB",
                                          family="JetBrains Mono, monospace"))
    tel_fig.update_layout(
        height=340,
        title=dict(text="Power Draw · kW", font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Time (simulated, accelerated 60×)",
        yaxis_title="Power (kW)", **PLOT_LAYOUT)

    water_fig = go.Figure()
    if not water.empty and 'building' in water.columns and 'water_m3' in water.columns:
        for i, b in enumerate(water.building.unique()):
            d = water[water.building == b]
            water_fig.add_trace(go.Scatter(
                x=d['ts'], y=d['water_m3'], name=f"Building {b}",
                mode='lines+markers',
                line=dict(color=colors[i % len(colors)], width=2.5),
                marker=dict(size=6, line=dict(color="#111A2C", width=1)),
                hovertemplate="<b>%{fullData.name}</b><br>"
                              "Time: %{x|%H:%M}<br>"
                              "Water: %{y:.2f} m³<extra></extra>"))
    else:
        water_fig.add_annotation(text="No water telemetry yet",
                                  xref="paper", yref="paper", x=0.5, y=0.5,
                                  showarrow=False,
                                  font=dict(size=13, color="#8794AB",
                                            family="JetBrains Mono, monospace"))
    water_fig.update_layout(
        height=340,
        title=dict(text="Water Consumption · m³", font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Time (simulated)", yaxis_title="Volume (m³)", **PLOT_LAYOUT)

    occ_fig = go.Figure()
    if not occ.empty and 'room_id' in occ.columns and 'count' in occ.columns:
        colors_bar = ["#F59E0B" if c == 0 else "#10B981" for c in occ['count']]
        sources = occ['source'] if 'source' in occ.columns else ["sim"] * len(occ)
        line_colors = ["#34D399" if s == "live" else "rgba(0,0,0,0)" for s in sources]
        source_labels = ["📷 live camera" if s == "live" else "simulated" for s in sources]
        occ_fig.add_trace(go.Bar(
            x=occ['room_id'], y=occ['count'],
            marker=dict(color=colors_bar,
                        line=dict(color=line_colors, width=3)),
            customdata=source_labels,
            hovertemplate="<b>%{x}</b><br>Occupancy: %{y} people<br>"
                          "Source: %{customdata}<extra></extra>"))
    else:
        occ_fig.add_annotation(text="No occupancy rows yet",
                                xref="paper", yref="paper", x=0.5, y=0.5,
                                showarrow=False,
                                font=dict(size=13, color="#8794AB",
                                          family="JetBrains Mono, monospace"))
    occ_fig.update_layout(
        height=340,
        title=dict(text="Live Occupancy by Room · amber = empty · emerald outline = webcam",
                   font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Room ID", yaxis_title="People counted", **PLOT_LAYOUT)

    return tel_fig, water_fig, occ_fig, badge


def ui_refresh_decide():
    try:
        pend, refs, alerts = get_decisions()
    except Exception:
        pend = refs = alerts = pd.DataFrame()

    ep = _empty_df(['id', 'room_id', 'device_id', 'state', 'expected_kw'])
    er = _empty_df(['ts', 'room_id', 'device_id', 'reason'])
    ea = _empty_df(['sent_at', 'room_id', 'priority', 'channel'])

    def _pick(df, cols, empty_df):
        if df is None or df.empty:
            return empty_df
        keep = [c for c in cols if c in df.columns]
        return df[keep] if keep else empty_df

    pend_df   = _pick(pend,   ['id', 'room_id', 'device_id', 'state', 'expected_kw'], ep)
    refs_df   = _pick(refs,   ['ts', 'room_id', 'device_id', 'reason'],                er)
    alerts_df = _pick(alerts, ['sent_at', 'room_id', 'priority', 'channel'],           ea)

    try:
        choices = _pending_choices()
    except Exception:
        choices = []

    return pend_df, refs_df, alerts_df, gr.update(choices=choices, value=None)


def ui_refresh_verify():
    try:
        led = get_verified()
    except Exception:
        led = _empty_df(['id', 'rec_id', 'room_id', 'device_id',
                         'delta_kw', 'co2_kg', 'confounder', 'created_at'])
    if led is None or led.empty:
        led = _empty_df(['id', 'rec_id', 'room_id', 'device_id',
                         'delta_kw', 'co2_kg', 'confounder', 'created_at'])

    try:
        tel, ghost, _, _ = get_live_data()
    except Exception:
        tel = ghost = pd.DataFrame()

    fig = go.Figure()
    b = "B1"
    if (not tel.empty and not ghost.empty
            and 'building' in tel.columns and 'building' in ghost.columns):
        b = tel.building.iloc[0]
        r = tel[tel.building == b]
        g = ghost[ghost.building == b]
        fig.add_trace(go.Scatter(
            x=r['ts'], y=r['kw'],
            name='LIVE TWIN (actions applied · savings)',
            line=dict(color='#10B981', width=3.5),
            hovertemplate="<b>Live</b><br>%{x|%H:%M}<br>%{y:.2f} kW<extra></extra>"))
        fig.add_trace(go.Scatter(
            x=g['ts'], y=g['ghost_kw'],
            name='GHOST TWIN (no actions · baseline)',
            line=dict(color='#F59E0B', width=3.5, dash='dash'),
            hovertemplate="<b>Ghost</b><br>%{x|%H:%M}<br>%{y:.2f} kW<extra></extra>"))
    else:
        fig.add_annotation(
            text="No live/ghost telemetry yet — approve a recommendation to populate",
            xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False,
            font=dict(size=13, color="#8794AB", family="JetBrains Mono, monospace"))
    fig.update_layout(
        height=440,
        title=dict(text=f"Counterfactual Verification · Building {b}",
                   font=dict(size=16, color="#FFFFFF")),
        xaxis_title="Time", yaxis_title="Power (kW)", **PLOT_LAYOUT)
    return led, fig


def ui_events():
    try:
        df = get_event_stream(25)
    except Exception:
        df = None
    if df is None or df.empty:
        return _empty_state(
            "No events yet.",
            "Events appear once the simulator generates its first anomaly.")
    rows = []
    for _, r in df.iterrows():
        ts = str(r["Timestamp"])[-8:] if r["Timestamp"] else "—"
        rows.append(
            f'<div class="ct-event">'
            f'<div class="ct-event-time">{ts}</div>'
            f'<div class="ct-event-kind kind-{r["Event"]}">{r["Event"]}</div>'
            f'<div class="ct-event-detail">{r["Detail"]}</div>'
            f'</div>')
    return f'<div class="ct-event-list">{"".join(rows)}</div>'


# ------- E-WASTE  (defensive rewrite) -------
_EW_DEFAULT = {
    "total_devices": 0, "total_kg": 0.0,
    "total_co2_embodied": 0.0, "total_co2_saved": 0.0,
    "total_revenue_inr": 0.0, "by_path": [], "by_department": [],
}

_EW_COLS = ['device_id','device_type','department','age_years',
            'condition','reason','kg_recoverable','recovery_path',
            'co2_saved_kg','revenue_inr']


def _safe_e_waste_summary():
    """Return a dict that always has every key ui_e_waste/ui_reuse_paths uses."""
    try:
        raw = e_waste_summary()
    except Exception as e:
        print("e_waste_summary() raised:", e)
        raw = {}
    if not isinstance(raw, dict):
        raw = {}
    out = dict(_EW_DEFAULT)
    for k, v in raw.items():
        out[k] = v
    # type-coerce numerics so f-strings with :.1f never blow up on None
    for k in ("total_devices","total_kg","total_co2_embodied",
              "total_co2_saved","total_revenue_inr"):
        try:
            out[k] = float(out[k] or 0)
        except Exception:
            out[k] = 0.0
    if not isinstance(out.get("by_path"), list):
        out["by_path"] = []
    if not isinstance(out.get("by_department"), list):
        out["by_department"] = []
    return out


def _safe_e_waste_table():
    """Return a DataFrame that always has the expected columns, no Nones crash."""
    try:
        c = conn()
        flags = pd.read_sql("""SELECT ef.device_id, ef.device_type, ef.department,
                               ef.age_years, ef.condition, ef.reason,
                               ef.kg_recoverable, er.recovery_path,
                               er.co2_saved_kg, er.revenue_inr
                               FROM e_waste_flags ef
                               LEFT JOIN e_waste_reuse er ON ef.device_id = er.device_id
                               ORDER BY er.co2_saved_kg DESC""", c)
        c.close()
    except Exception as e:
        print("e_waste query raised:", e)
        flags = pd.DataFrame(columns=_EW_COLS)

    if flags is None or flags.empty:
        return pd.DataFrame(columns=_EW_COLS)

    # ensure every column is present, in order
    for col in _EW_COLS:
        if col not in flags.columns:
            flags[col] = None
    flags = flags[_EW_COLS].copy()

    # fill numeric NaNs, stringify text columns → consistent dtypes for Gradio
    for col in ("age_years","kg_recoverable","co2_saved_kg","revenue_inr"):
        flags[col] = pd.to_numeric(flags[col], errors="coerce").fillna(0)
    for col in ("device_id","device_type","department",
                "condition","reason","recovery_path"):
        flags[col] = flags[col].fillna("").astype(str)
    return flags


def ui_e_waste():
    ew = _safe_e_waste_summary()
    flags = _safe_e_waste_table()
    summary = f"""
    <div class="ct-metrics">
      {_metric_card(int(ew['total_devices']), "Devices Flagged",
                    "Past EOL, non-functional, or unserviced", "alert")}
      {_metric_card(f"{ew['total_kg']:.1f}", "Material Recovery",
                    "Total kg recoverable across all devices", "warn", "kg")}
      {_metric_card(f"{ew['total_co2_saved']:.0f}", "CO₂ Avoided by Reuse",
                    "Embodied carbon retained vs. new device", "success", "kg")}
      {_metric_card(f"₹{ew['total_revenue_inr']:.0f}", "Recovery Revenue",
                    "Value from refurbishment + parts", "success")}
    </div>
    """
    return summary, flags


def ui_reuse_paths():
    ew = _safe_e_waste_summary()
    paths = ew.get("by_path") or []
    fig = go.Figure()
    if paths:
        xs, ys, labels = [], [], []
        for p in paths:
            try:
                xs.append(str(p.get("path", "—")))
                ys.append(float(p.get("co2_saved_kg", 0) or 0))
                labels.append(f"{float(p.get('co2_saved_kg', 0) or 0):.0f} kg"
                              f"<br><span style='font-size:10px'>"
                              f"{int(p.get('count', 0) or 0)} devices</span>")
            except Exception:
                continue
        fig.add_trace(go.Bar(
            x=xs, y=ys,
            marker_color="#10B981",
            marker_line_color="#34D399", marker_line_width=1.5,
            text=labels, textposition="outside",
            textfont=dict(color="#FFFFFF", size=11),
            hovertemplate="<b>%{x}</b><br>CO₂ avoided: %{y:.0f} kg<extra></extra>"))
    else:
        fig.add_annotation(
            text="No e-waste flagged yet — click Re-scan Inventory",
            xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False,
            font=dict(size=13, color="#8794AB", family="JetBrains Mono, monospace"))
    fig.update_layout(
        height=400,
        title=dict(text="E-Waste Recovery Pathways · CO₂ Avoided",
                   font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Recovery Path", yaxis_title="CO₂ avoided (kg)",
        showlegend=False, **PLOT_LAYOUT)
    return fig


def ui_carbon():
    try:
        cm = pickle.load(open(f"{MODEL_DIR}/carbon_forecast.pkl", "rb"))
    except Exception:
        return (_placeholder_fig("Daily CO₂ — Historical + 14-Day Forecast",
                                 "Carbon model not trained"),
                _empty_df(["date", "predicted_co2_kg"]),
                _empty_state("Model not trained."))
    try:
        c = conn()
        tel = pd.read_sql("SELECT ts, kw FROM telemetry ORDER BY ts DESC LIMIT 200", c)
        c.close()
    except Exception:
        tel = pd.DataFrame()
    if tel.empty:
        return (_placeholder_fig("Daily CO₂ — Historical + 14-Day Forecast",
                                 "No telemetry yet"),
                _empty_df(["date", "predicted_co2_kg"]),
                _empty_state("No telemetry yet."))
    tel['ts'] = pd.to_datetime(tel['ts'])
    tel['date'] = tel['ts'].dt.date
    daily = tel.groupby('date')['kw'].sum().reset_index()
    daily['date'] = pd.to_datetime(daily['date'])
    daily['co2_kg'] = daily['kw'] * 0.71
    daily = daily.sort_values('date').reset_index(drop=True)
    last = daily.iloc[-1]
    prev, prev7 = last['co2_kg'], daily['co2_kg'].tail(7).mean()
    future = []
    for i in range(1, 15):
        d = last['date'] + pd.Timedelta(days=i)
        feat = pd.DataFrame([{'doy': d.dayofyear, 'dow': d.weekday(),
                              'is_weekend': int(d.weekday() >= 5),
                              'lag1': prev, 'lag7': prev, 'roll7': prev7}])[
            ['doy','dow','is_weekend','lag1','lag7','roll7']]
        p = max(0, float(cm.predict(feat)[0]))
        future.append({'date': d, 'predicted_co2_kg': p})
        prev = p
    fdf = pd.DataFrame(future)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=daily['date'], y=daily['co2_kg'], name='Historical',
        mode='lines+markers', line=dict(color='#10B981', width=3),
        marker=dict(size=6),
        hovertemplate="<b>%{x|%d %b}</b><br>Actual: %{y:.0f} kg<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=fdf['date'], y=fdf['predicted_co2_kg'], name='14-day Forecast',
        mode='lines+markers', line=dict(color='#3B82F6', dash='dash', width=3),
        marker=dict(size=6),
        hovertemplate="<b>%{x|%d %b}</b><br>Forecast: %{y:.0f} kg<extra></extra>"))
    fig.update_layout(
        height=440,
        title=dict(text="Daily CO₂ — Historical + 14-Day Forecast",
                   font=dict(size=16, color="#FFFFFF")),
        xaxis_title="Date", yaxis_title="CO₂ emissions (kg/day)", **PLOT_LAYOUT)

    total_co2 = float(daily['co2_kg'].sum()) + float(fdf['predicted_co2_kg'].sum())
    eq = carbon_equivalents(total_co2)
    ew = _safe_e_waste_summary()
    ew_co2 = ew.get('total_co2_saved', 0) or 0
    combined = eq['trees_years'] + ew_co2 / 21.0
    eq_html = f"""
    <div class="ct-metrics">
      {_metric_card(f"{eq['trees_years']}", "Tree-Years (Energy)",
                    "Trees needed to absorb energy CO₂ for a year", "success")}
      {_metric_card(f"{ew_co2/21:.1f}", "Tree-Years (E-Waste)",
                    "Additional from device reuse", "success")}
      {_metric_card(f"{eq['km_driven']:,.0f}", "km of Driving",
                    "Equivalent car travel avoided", "info", "km")}
      {_metric_card(f"{eq['led_bulb_hours']:,}", "LED Bulb Hours",
                    "Runtime of a 9W LED from saved energy", "info")}
      {_metric_card(f"{combined:.1f}", "Combined Tree-Years",
                    "Energy + e-waste reuse total", "success")}
    </div>
    """
    return fig, fdf, eq_html


def ui_allocation():
    try:
        res = run_optimizer()
    except Exception as e:
        return (_empty_state(f"Solver error: {e}"), _empty_df(["Session"]),
                _placeholder_fig("Optimization Impact"))
    if "error" in res:
        return (_empty_state(f"Solver {res['error']}"), _empty_df(["Session"]),
                _placeholder_fig("Optimization Impact",
                                 "Solver did not return a feasible plan"))
    summary = f"""
    <div class="ct-metrics">
      {_metric_card(res['status'], "Solver Status",
                    "OPTIMAL = proven best solution", "info")}
      {_metric_card(f"{res['solve_ms']:.0f}", "Solve Time",
                    "CP-SAT wall time in milliseconds", "muted", "ms")}
      {_metric_card(res['before_waste_seats'], "Wasted Seats (Before)",
                    "Unused capacity across all sessions", "warn")}
      {_metric_card(res['after_waste_seats'], "Wasted Seats (After)",
                    "Optimal allocation reduces this", "success")}
      {_metric_card(f"−{res['waste_reduction_pct']}", "Waste Reduction",
                    "Fewer empty seats, better utilization", "success", "%")}
      {_metric_card(f"−{res['energy_reduction_pct']}", "Energy Reduction",
                    "Fewer rooms powered at the same time", "success", "%")}
    </div>
    """
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Before (baseline)", x=["Wasted seats", "Energy (kWh)"],
        y=[res["before_waste_seats"], res["before_energy_kwh"]],
        marker_color="#F59E0B", marker_line_color="#FBBF24", marker_line_width=1.5,
        text=[f"{res['before_waste_seats']} seats", f"{res['before_energy_kwh']:.0f} kWh"],
        textposition="outside", textfont=dict(color="#FFFFFF", size=12),
        hovertemplate="<b>Before</b><br>%{x}: %{y}<extra></extra>"))
    fig.add_trace(go.Bar(
        name="After (optimized)", x=["Wasted seats", "Energy (kWh)"],
        y=[res["after_waste_seats"], res["after_energy_kwh"]],
        marker_color="#10B981", marker_line_color="#34D399", marker_line_width=1.5,
        text=[f"{res['after_waste_seats']} seats", f"{res['after_energy_kwh']:.0f} kWh"],
        textposition="outside", textfont=dict(color="#FFFFFF", size=12),
        hovertemplate="<b>After</b><br>%{x}: %{y}<extra></extra>"))
    fig.update_layout(
        height=360, barmode="group",
        title=dict(text="Optimization Impact · Before vs After",
                   font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Metric", yaxis_title="Value", **PLOT_LAYOUT)
    return summary, res["assignments"], fig


# ------- F1  (defensive rewrite — dynamic key handling) -------
# (key, label, style, help_text) — in the order we want them displayed.
_F1_FIELDS = [
    ('f1',                          'F1 Score',        'success', 'Balance of precision and recall'),
    ('precision',                   'Precision',       'info',    'Fraction of alerts that were true positives'),
    ('recall',                      'Recall',          'info',    'Fraction of true anomalies we caught'),
    ('accuracy',                    'Accuracy',        'info',    'Overall correctness across all ticks'),
    ('true_positives',              'True Positives',  'success', 'Empty-room ticks correctly flagged'),
    ('true_negatives',              'True Negatives',  'success', 'Occupied-room ticks correctly left alone'),
    ('false_positives',             'False Positives', 'alert',   'Occupied rooms flagged anyway'),
    ('false_negatives',             'False Negatives', 'alert',   'Empty-room waste we missed'),
    ('gt_positives_empty_room_ticks','GT Positives',   'muted',   'Empty-room ticks in the occupancy log'),
    ('gt_negatives_occupied_room_ticks','GT Negatives','muted',   'Occupied-room ticks in the occupancy log'),
]


def ui_f1():
    try:
        res = run_adversarial_test()
    except Exception as e:
        err = traceback.format_exc()
        print("run_adversarial_test failed:\n", err)
        return _empty_state(f"Test error: {e}"), _empty_df(["Metric", "Value"])

    if not isinstance(res, dict):
        return (_empty_state("Test returned unexpected value"),
                _empty_df(["Metric", "Value"]))

    # Build only the cards whose keys are actually present.
    cards = []
    for key, label, style, help_txt in _F1_FIELDS:
        if key in res:
            v = res[key]
            # Format numbers nicely, but leave strings alone
            if isinstance(v, float):
                v_fmt = f"{v:.3f}" if abs(v) < 10 else f"{v:.1f}"
            elif isinstance(v, int):
                v_fmt = f"{v:,}"
            else:
                v_fmt = str(v)
            cards.append(_metric_card(v_fmt, label, help_txt, style))

    # Fallback: whatever keys exist, show them
    if not cards:
        for k, v in res.items():
            if k == 'ground_truth':
                continue
            cards.append(_metric_card(str(v), k.replace('_', ' ').title(), '', 'info'))

    # Extra info line: if the function returned a ground_truth string, note it
    gt_note = ""
    if isinstance(res.get("ground_truth"), str):
        gt_note = f'<div class="ct-caveat">ℹ Ground truth source: <code>{res["ground_truth"]}</code></div>'

    summary = f'{gt_note}<div class="ct-metrics">{"".join(cards)}</div>'

    df = pd.DataFrame([{"Metric": k.replace("_", " ").title(), "Value": v}
                       for k, v in res.items()])
    return summary, df


def ui_departments():
    try:
        depts = get_department_summary()
    except Exception:
        depts = {}
    if not depts:
        return (_empty_state("No data yet. Approve some recommendations first."),
                _empty_df(["Department"]),
                _placeholder_fig("CO₂ Impact per Department"),
                _placeholder_fig("Policy Refusals by Department"))
    rows = sorted(depts.values(), key=lambda x: -x.get("kwh", 0))
    total_kwh = sum(d.get("kwh", 0) for d in rows)
    total_co2 = sum(d.get("co2", 0) for d in rows)
    total_ewaste = sum(d.get("e_waste_co2", 0) for d in rows)
    top = rows[0] if rows else None

    metrics = f"""
    <div class="ct-metrics">
      {_metric_card(len(rows), "Departments Tracked",
                    "Each room maps to a department", "info")}
      {_metric_card(f"{total_kwh:.2f}", "Verified kWh",
                    "Energy savings confirmed via ghost twin", "success", "kWh")}
      {_metric_card(f"{total_co2:.1f}", "Energy CO₂",
                    "Grid emissions avoided (kg)", "success", "kg")}
      {_metric_card(f"{total_ewaste:.0f}", "E-Waste CO₂",
                    "Estimate — not summed with energy CO₂", "info", "kg")}
      {_metric_card(top['department'] if top else '—', "Top Contributor",
                    "Highest verified energy kWh", "warn")}
    </div>
    """

    df = pd.DataFrame([{
        "Department": d.get("department", "—"), "Rooms": d.get("rooms", 0),
        "Verified kWh": round(d.get("kwh", 0), 2),
        "Energy CO₂ (kg)": round(d.get("co2", 0), 2),
        "E-Waste Devices": d.get("e_waste_devices", 0),
        "E-Waste (kg)": d.get("e_waste_kg", 0),
        "E-Waste CO₂ (kg)": d.get("e_waste_co2", 0),
        "Pending": d.get("pending", 0), "Refused": d.get("refused", 0),
    } for d in rows])

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=[d.get("department", "—") for d in rows],
        y=[d.get("co2", 0) for d in rows], name="Energy CO₂ (measured)",
        marker_color="#10B981", marker_line_color="#34D399", marker_line_width=1.5,
        text=[f"{d.get('co2', 0):.1f}" for d in rows], textposition="outside",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="<b>%{x}</b><br>Energy CO₂: %{y:.1f} kg<extra></extra>"))
    fig1.add_trace(go.Bar(
        x=[d.get("department", "—") for d in rows],
        y=[d.get("e_waste_co2", 0) for d in rows], name="E-Waste Reuse CO₂ (estimated)",
        marker_color="#3B82F6", marker_line_color="#60A5FA", marker_line_width=1.5,
        text=[f"{d.get('e_waste_co2', 0):.0f}" for d in rows], textposition="outside",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="<b>%{x}</b><br>E-waste CO₂: %{y:.0f} kg<extra></extra>"))
    fig1.update_layout(
        height=420, barmode="group",
        title=dict(text="CO₂ Impact per Department · Energy (measured) vs E-Waste (estimated)",
                   font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Department", yaxis_title="CO₂ avoided (kg)", **PLOT_LAYOUT)

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=[d.get("department", "—") for d in rows],
        y=[d.get("refused", 0) for d in rows], name="Refusals",
        marker_color="#F59E0B", marker_line_color="#FBBF24", marker_line_width=1.5,
        text=[str(d.get("refused", 0)) for d in rows], textposition="outside",
        textfont=dict(color="#FFFFFF", size=11),
        hovertemplate="<b>%{x}</b><br>Refusals: %{y}<extra></extra>"))
    fig2.update_layout(
        height=420,
        title=dict(text="Policy Refusals by Department",
                   font=dict(size=15, color="#FFFFFF")),
        xaxis_title="Department", yaxis_title="Number of refusals",
        showlegend=False, **PLOT_LAYOUT)

    return metrics, df, fig1, fig2


# ==========================================================================
# ROBUST REPORT GENERATION
# ==========================================================================
def _find_writable_dir():
    for cand in ["/kaggle/working", os.getcwd(), "/tmp", tempfile.gettempdir()]:
        try:
            os.makedirs(cand, exist_ok=True)
            probe = os.path.join(cand, ".__rt_probe")
            with open(probe, "w") as fh:
                fh.write("x")
            os.remove(probe)
            return cand
        except Exception:
            continue
    return None


def ui_report():
    try:
        md, md_path, json_path = ui_report_v2()
        if not md_path or not os.path.exists(str(md_path)):
            raise FileNotFoundError(f"Markdown report path missing: {md_path!r}")
        if not json_path or not os.path.exists(str(json_path)):
            raise FileNotFoundError(f"JSON report path missing: {json_path!r}")
        return md, md_path, json_path
    except Exception as e:
        err = traceback.format_exc()
        print("=" * 70)
        print("REPORT GENERATION FAILED")
        print("=" * 70)
        print(err)
        print("=" * 70)

        err_md = (
            f"# Report generation failed\n\n"
            f"**{type(e).__name__}:** `{e}`\n\n"
            f"### Traceback\n\n"
            f"```\n{err}\n```\n"
        )

        out_dir = _find_writable_dir()
        if out_dir is None:
            return err_md, None, None

        ts = datetime.now().strftime("%H%M%S")
        md_path = os.path.join(out_dir, f"report_error_{ts}.md")
        json_path = os.path.join(out_dir, f"report_error_{ts}.json")
        try:
            with open(md_path, "w", encoding="utf-8") as fh:
                fh.write(err_md)
            with open(json_path, "w", encoding="utf-8") as fh:
                json.dump({"error": str(e),
                           "type": type(e).__name__,
                           "traceback": err}, fh, indent=2)
            return err_md, md_path, json_path
        except Exception as e2:
            print("Fallback write also failed:", e2)
            return err_md + f"\n\nFallback write failed: `{e2}`", None, None


# ==========================================================================
# PRE-COMPUTE INITIAL STATE
# ==========================================================================
print("Pre-computing initial UI state…")
try:
    _init_tel_fig, _init_water_fig, _init_occ_fig, _init_live_badge = ui_refresh_live(None)
except Exception as e:
    print("  live init:", e)
    _init_tel_fig = _placeholder_fig("Power Draw · kW")
    _init_water_fig = _placeholder_fig("Water Consumption · m³")
    _init_occ_fig = _placeholder_fig("Live Occupancy by Room")
    _init_live_badge = _empty_state("Camera inactive",
                                     "Click the webcam box and allow permission")

try:
    _init_pend_df, _init_refs_df, _init_alerts_df, _ = ui_refresh_decide()
except Exception as e:
    print("  decide init:", e)
    _init_pend_df = _empty_df(['id','room_id','device_id','state','expected_kw'])
    _init_refs_df = _empty_df(['ts','room_id','device_id','reason'])
    _init_alerts_df = _empty_df(['sent_at','room_id','priority','channel'])

try:
    _init_pend_choices = _pending_choices()
except Exception:
    _init_pend_choices = []

try:
    _init_ledger, _init_verify_fig = ui_refresh_verify()
except Exception as e:
    print("  verify init:", e)
    _init_ledger = _empty_df(['id','rec_id','room_id','device_id',
                              'delta_kw','co2_kg','confounder','created_at'])
    _init_verify_fig = _placeholder_fig("Counterfactual Verification")

try:
    _init_events_html = ui_events()
except Exception:
    _init_events_html = _empty_state("No events yet.")

try:
    _init_ew_sum, _init_ew_tbl = ui_e_waste()
    _init_ew_chart = ui_reuse_paths()
except Exception as e:
    print("  e-waste init:", e)
    _init_ew_sum = _empty_state("Not scanned")
    _init_ew_tbl = _empty_df(_EW_COLS)
    _init_ew_chart = _placeholder_fig("E-Waste Recovery Pathways")

try:
    _init_dept_metrics, _init_dept_tbl, _init_dept_fig1, _init_dept_fig2 = ui_departments()
except Exception as e:
    print("  dept init:", e)
    _init_dept_metrics = _empty_state("No summary yet",
                                       "Click Refresh to aggregate all departments")
    _init_dept_tbl = _empty_df(["Department"])
    _init_dept_fig1 = _placeholder_fig("CO₂ Impact per Department")
    _init_dept_fig2 = _placeholder_fig("Policy Refusals by Department")

print("  done.")


# ==========================================================================
# BUILD UI
# ==========================================================================
with gr.Blocks(title="Campus Twin AI", theme=gr.themes.Base()) as demo:
    gr.HTML(f"<style>{CUSTOM_CSS}</style>")
    gr.HTML(HEADER_HTML)

    with gr.Row():
        metrics_html = gr.HTML(ui_metrics())
        refresh_metrics_btn = gr.Button("↻ Refresh Metrics", variant="secondary", scale=0)
    refresh_metrics_btn.click(ui_metrics, outputs=[metrics_html])

    with gr.Tabs():

        # ---------- LIVE ----------
        with gr.Tab("Live"):
            gr.HTML(_tab_intro(
                "Live Campus State",
                "Real-time view of the twin simulator. The <code>YOLOv8n</code> model streams "
                "detection from your webcam — counts are written to the same <code>occupancy</code> "
                "table the simulator uses, for the one room the camera represents. Every other room "
                "across the simulated campus uses a synthetic timetable to demonstrate the system at "
                "campus scale; only this one room's occupancy comes from a live camera today. Power "
                "and water charts replay historical BDG2 data at 60× speed."))
            gr.HTML(_legend([
                ("Power (kW)", "#10B981"),
                ("Water (m³)", "#3B82F6"),
                ("Occupied room", "#10B981"),
                ("Empty room", "#F59E0B"),
            ]))

            with gr.Row():
                with gr.Column(scale=3):
                    gr.HTML('<div class="ct-section-label">Live Camera · YOLOv8n Object Detection '
                            '<span class="hint">— click the box below, allow camera access</span></div>')
                    cam = gr.Image(sources=["webcam"], streaming=True, type="numpy",
                                    label="Live feed · click to activate",
                                    height=320, show_label=True)
                    live_badge = gr.HTML(_init_live_badge)
                    with gr.Row():
                        snap_btn = gr.Button("📸 Capture Snapshot", variant="secondary")
                        stream_toggle_btn = gr.Button("⏸ Pause Stream", variant="secondary")
                    stream_status = gr.HTML(_inline_status("Stream ready"))

                with gr.Column(scale=2):
                    gr.HTML('<div class="ct-section-label">Simulator Controls '
                            '<span class="hint">— the twin runs every 2s</span></div>')
                    pause_btn = gr.Button("⏸ Pause / Resume Simulator", variant="secondary")
                    pause_status = gr.HTML(_inline_status("Running · 2.0 s/tick"))
                    speed_slider = gr.Slider(0.5, 5.0, value=2.0, step=0.5,
                                              label="Tick Interval",
                                              info="Seconds per simulated hour — lower is faster")
                    inject_btn = gr.Button("⚡ Inject Anomaly Now", variant="secondary")
                    inject_status = gr.HTML(_inline_status("Ready to inject", "muted"))
                    reset_btn = gr.Button("↻ Reset Sim Hour", variant="secondary")
                    reset_status = gr.HTML(_inline_status("Ready to reset", "muted"))

                    def _wrap_html(fn):
                        def wrapped(*a, **kw):
                            try:
                                return _inline_status(fn(*a, **kw))
                            except Exception as e:
                                return _inline_status(f"Error: {e}", "warn")
                        return wrapped

                    pause_btn.click(_wrap_html(toggle_pause), outputs=pause_status)
                    speed_slider.change(_wrap_html(set_speed), inputs=speed_slider, outputs=pause_status)
                    inject_btn.click(_wrap_html(inject_anomaly), outputs=inject_status)
                    reset_btn.click(_wrap_html(reset_sim), outputs=reset_status)

            gr.HTML('<div class="ct-section-label">Telemetry Streams '
                    '<span class="hint">— hover any point for exact values</span></div>')
            with gr.Row():
                tel_plot = gr.Plot(value=_init_tel_fig)
                water_plot = gr.Plot(value=_init_water_fig)
            occ_plot = gr.Plot(value=_init_occ_fig)

            gr.HTML('<div class="ct-section-label">Live Event Stream '
                    '<span class="hint">— recommendations, refusals, verifications, alerts</span></div>')
            events_html = gr.HTML(_init_events_html)
            events_btn = gr.Button("Refresh Events", variant="secondary")
            events_btn.click(ui_events, outputs=events_html)

            cam.stream(run_yolo_stream, inputs=cam, outputs=[cam, live_badge],
                       show_progress="hidden")
            snap_btn.click(ui_refresh_live, inputs=cam,
                           outputs=[tel_plot, water_plot, occ_plot, live_badge])

            def _toggle_stream_wrap():
                try:
                    return _inline_status(toggle_stream())
                except Exception as e:
                    return _inline_status(f"Error: {e}", "warn")
            stream_toggle_btn.click(_toggle_stream_wrap, outputs=stream_status)

        # ---------- DECIDE ----------
        with gr.Tab("Decide"):
            gr.HTML(_tab_intro(
                "Human-in-the-Loop Decision Gate",
                "Every AI recommendation is queued here as <code>PENDING</code>. Nothing executes "
                "until a human clicks Approve. Refusals are logged with their reason — policy is a feature."))

            gr.HTML('<div class="ct-section-label">Pending Recommendations '
                    '<span class="hint">— pick one from the dropdown, click Approve or Reject</span></div>')
            pend_tbl = gr.Dataframe(value=_init_pend_df, label="Queue",
                                    show_label=True, wrap=False)
            with gr.Row():
                rec_id_in = gr.Dropdown(label="Pending recommendation",
                                        choices=_init_pend_choices, value=None,
                                        interactive=True, scale=3)
                approve_btn = gr.Button("✓ Approve", variant="primary", scale=1)
                reject_btn = gr.Button("✕ Reject", variant="secondary", scale=1)
            action_out = gr.HTML(_inline_status("Awaiting your decision", "muted"))

            def _approve_and_refresh(rid):
                try:
                    msg = approve_rec(rid)
                except Exception as e:
                    msg = f"Error: {e}"
                try:
                    choices = _pending_choices()
                except Exception:
                    choices = []
                return _inline_status(msg), gr.update(choices=choices, value=None)

            def _reject_and_refresh(rid):
                try:
                    msg = reject_rec(rid)
                except Exception as e:
                    msg = f"Error: {e}"
                try:
                    choices = _pending_choices()
                except Exception:
                    choices = []
                return _inline_status(msg, "warn"), gr.update(choices=choices, value=None)

            approve_btn.click(_approve_and_refresh, inputs=rec_id_in,
                              outputs=[action_out, rec_id_in])
            reject_btn.click(_reject_and_refresh, inputs=rec_id_in,
                             outputs=[action_out, rec_id_in])

            with gr.Row():
                with gr.Column():
                    gr.HTML('<div class="ct-section-label">Refusals '
                            '<span class="hint">— actions we deliberately did not take</span></div>')
                    ref_tbl = gr.Dataframe(value=_init_refs_df, label="Refusals",
                                            show_label=True, wrap=False)
                with gr.Column():
                    gr.HTML('<div class="ct-section-label">Alerts Dispatched '
                            '<span class="hint">— Telegram, email, or in-app</span></div>')
                    alert_tbl = gr.Dataframe(value=_init_alerts_df, label="Alerts",
                                              show_label=True, wrap=False)

            refresh_decide_btn = gr.Button("↻ Refresh Queue", variant="secondary")
            refresh_decide_btn.click(
                ui_refresh_decide,
                outputs=[pend_tbl, ref_tbl, alert_tbl, rec_id_in])

        # ---------- DEPARTMENTS ----------
        with gr.Tab("Departments"):
            gr.HTML(_tab_intro(
                "Department Accountability",
                "Every verified action is attributed to a department via the room registry. "
                "E-waste reuse is joined per department. The chart shows energy CO₂ (measured) "
                "and e-waste CO₂ (an assumption-based estimate) side by side, grouped rather "
                "than stacked, so the two are never visually or numerically summed into one figure."))

            dept_btn = gr.Button("↻ Refresh Department Summary", variant="primary")
            dept_metrics = gr.HTML(_init_dept_metrics)
            dept_tbl = gr.Dataframe(value=_init_dept_tbl, label="Department Breakdown",
                                     show_label=True, wrap=False)
            with gr.Row():
                dept_chart = gr.Plot(value=_init_dept_fig1)
                dept_refused_chart = gr.Plot(value=_init_dept_fig2)
            dept_btn.click(ui_departments,
                           outputs=[dept_metrics, dept_tbl, dept_chart, dept_refused_chart])

        # ---------- VERIFY ----------
        with gr.Tab("Verify"):
            gr.HTML(_tab_intro(
                "Counterfactual Verification",
                "The <code>LIVE TWIN</code> applies recommendations. The <code>GHOST TWIN</code> "
                "runs the same simulation but never applies them. Both are derived from the same "
                "historical baseline, so the gap between the two lines demonstrates the "
                "verification <i>mechanism</i> we would apply to real smart-meter data. On this "
                "twin, the OFF effect is perturbed by ±15% each tick rather than subtracted "
                "exactly, so the observed delta is a noisy estimate centered on the assumed "
                "device wattage. Swapping in live meters would make this a genuine causal "
                "measurement with no other code changes."))

            gr.HTML(_legend([
                ("Live Twin — actions applied (savings)", "#10B981"),
                ("Ghost Twin — no actions (baseline)", "#F59E0B"),
            ]))

            verify_fig = gr.Plot(value=_init_verify_fig)
            gr.HTML('<div class="ct-section-label">Verified Ledger '
                    '<span class="hint">— every approved action with its measured delta</span></div>')
            ledger_tbl = gr.Dataframe(value=_init_ledger, label="Ledger",
                                       show_label=True, wrap=False)
            v_btn = gr.Button("↻ Refresh Verification", variant="secondary")
            v_btn.click(ui_refresh_verify, outputs=[ledger_tbl, verify_fig])

        # ---------- ALLOCATION ----------
        with gr.Tab("Allocation"):
            gr.HTML(_tab_intro(
                "Multi-Objective Room Allocation",
                "CP-SAT solver balances five objectives: <b>wasted seats</b>, <b>energy</b>, "
                "<b>relocations</b>, <b>peak load</b>, and <b>department affinity</b>. "
                "Hard constraints: capacity, room type, no double-booking."))

            alloc_btn = gr.Button("▶ Re-plan Rooms", variant="primary")
            alloc_sum = gr.HTML(_empty_state(
                "Solver idle",
                "Click Re-plan to run CP-SAT on the scheduled sessions"))
            alloc_tbl = gr.Dataframe(label="Assignment Table", show_label=True, wrap=False)
            alloc_chart = gr.Plot(value=_placeholder_fig(
                "Optimization Impact · Before vs After",
                "Click Re-plan Rooms to solve"))
            alloc_btn.click(ui_allocation, outputs=[alloc_sum, alloc_tbl, alloc_chart])

        # ---------- E-WASTE ----------
        with gr.Tab("E-Waste"):
            gr.HTML(_tab_intro(
                "E-Waste Inventory & Reuse Planning",
                "Every device in the registry is scanned for EOL age, condition, or stale servicing. "
                "The reuse engine then assigns a recovery path with its own CO₂ recovery percentage "
                "and revenue estimate."))

            gr.HTML(_legend([
                ("Refurbish (70% CO₂ recovery)", "#10B981"),
                ("Donate (65%)", "#3B82F6"),
                ("Parts harvest (40%)", "#8B5CF6"),
                ("Certified recycle (15%)", "#F59E0B"),
            ]))

            ew_btn = gr.Button("↻ Re-scan Inventory", variant="primary")
            ew_sum = gr.HTML(_init_ew_sum)
            ew_chart = gr.Plot(value=_init_ew_chart)
            ew_tbl = gr.Dataframe(value=_init_ew_tbl, label="Device Inventory",
                                   show_label=True, wrap=False)

            def refresh_ew():
                s, t = ui_e_waste()
                fig = ui_reuse_paths()
                return s, t, fig

            ew_btn.click(refresh_ew, outputs=[ew_sum, ew_tbl, ew_chart])

        # ---------- SUSTAINABILITY ----------
        with gr.Tab("Sustainability"):
            gr.HTML(_tab_intro(
                "Carbon Forecast & Equivalents",
                "<code>LightGBM</code> trained on 585 days of historical BDG2 data. "
                "14-day forward forecast with MAE ~350 kg / day (MAPE ~3%). "
                "Carbon equivalents translate kg CO₂ into relatable units."))

            carbon_btn = gr.Button("▶ Run Forecast", variant="primary")
            carbon_fig = gr.Plot(value=_placeholder_fig(
                "Daily CO₂ — Historical + 14-Day Forecast",
                "Click Run Forecast to render historical + forecast"))
            carbon_eq = gr.HTML()
            carbon_tbl = gr.Dataframe(label="Forecast Values", show_label=True, wrap=False)
            carbon_btn.click(ui_carbon, outputs=[carbon_fig, carbon_tbl, carbon_eq])

        # ---------- EVIDENCE ----------
        with gr.Tab("Evidence"):
            gr.HTML(_tab_intro(
                "Model Evidence & Metrics",
                "Every number here was measured on a holdout set. No confidence scores are "
                "presented as proof. Click <b>Run F1 Test</b> to score the anomaly detector "
                "against every occupancy tick the simulator has logged: an empty-room tick is "
                "the positive class (waste condition), an occupied-room tick is the negative "
                "class (nothing to flag) — a real precision/recall test with true negatives. "
                "It's still a self-consistency check against this simulation's own occupancy "
                "log, disclosed as such, not validation against independent external ground truth."))

            f1_btn = gr.Button("▶ Run Adversarial F1 Test", variant="primary")
            f1_sum = gr.HTML(_empty_state(
                "Not scored",
                "Click to score against the logged occupancy history"))
            f1_tbl = gr.Dataframe(label="Confusion Matrix", show_label=True, wrap=False)
            f1_btn.click(ui_f1, outputs=[f1_sum, f1_tbl])

            gr.HTML('<div class="ct-section-label">Energy Forecast Metrics '
                    '<span class="hint">— time-based holdout, 90/10 split</span></div>')
            try:
                _m = json.load(open(f"{MODEL_DIR}/metrics.json"))
                gr.Dataframe(
                    value=pd.DataFrame([{"Building": k, **v} for k, v in _m.items()]),
                    label="Energy · MAPE / MAE / residual σ", show_label=True, wrap=False)
            except Exception:
                gr.HTML(_empty_state("Metrics not available."))

            gr.HTML('<div class="ct-section-label">Water Forecast Metrics '
                    '<span class="hint">— MAPE computed only over hours with water > 0.01 m³</span></div>')
            try:
                _wm = json.load(open(f"{MODEL_DIR}/water_metrics.json"))
                dead = [b for b, v in _wm.items() if v.get("nonzero_coverage_pct", 100) <= 1]
                sparse = [b for b, v in _wm.items()
                          if 1 < v.get("nonzero_coverage_pct", 100) < 80]
                ok = [b for b in _wm if b not in dead and b not in sparse]
                caveat_bits = []
                if dead:
                    caveat_bits.append(f"{', '.join(dead)} — no usable readings (dead meter)")
                if sparse:
                    caveat_bits.append(f"{', '.join(sparse)} — significant gaps in coverage")
                if caveat_bits:
                    ok_note = (f" Only {', '.join(ok)} "
                               + ("is" if len(ok) == 1 else "are")
                               + " used operationally.") if ok else ""
                    gr.HTML(f'<div class="ct-caveat">⚠ {"; ".join(caveat_bits)}.{ok_note}</div>')
                gr.Dataframe(
                    value=pd.DataFrame([{"Building": k, **v} for k, v in _wm.items()]),
                    label="Water · MAPE / MAE / coverage", show_label=True, wrap=False)
            except Exception:
                pass

        # ---------- REPORT ----------
        with gr.Tab("Report"):
            gr.HTML(_tab_intro(
                "Session Report",
                "Generates a Markdown + JSON report combining energy savings, e-waste reuse, "
                "department accountability, and refusal breakdown. The executive summary uses "
                "a template fallback (works offline) with optional LLM enrichment."))

            gen_btn = gr.Button("📄 Generate Session Report", variant="primary")
            report_md = gr.Markdown()
            with gr.Row():
                report_file_md = gr.File(label="Download · Markdown",
                                         file_count="single", interactive=False)
                report_file_json = gr.File(label="Download · JSON",
                                            file_count="single", interactive=False)
            gen_btn.click(ui_report,
                          outputs=[report_md, report_file_md, report_file_json])

    gr.HTML(FOOTER_HTML)

demo.launch(share=True, debug=False)

Pre-computing initial UI state…
  done.


/tmp/ipykernel_58/435957392.py:1368: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://fb4ee133970952201b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
